# MemoryFearNetwork MVPA Output Visualization (p < .01 Importance Mask)
Ordered report for saved outputs from the `mvpa_L2_*` scripts. The notebook loads checkpoint/intermediate files from `/Users/xiaoqianxiao/projects/NARSAD/LSS/results/MemoryFearNetwork`. This variant reconstructs stage-11 importance masks using saved permutation p-values at `p < .01`; sections 4+ rebuild the main downstream visualization tables in memory with `ACTIVE_IMPORTANCE_MASKS`, while leaving saved `.joblib` checkpoints unchanged.


## Setup


In [ ]:
import os
import math
import glob
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Image, Markdown
from scipy import stats
import statsmodels.api as sm

warnings.filterwarnings("ignore", category=RuntimeWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300})

PIPELINE_NAME = "MemoryFearNetwork"
PIPELINE_LABEL = "MemoryFearNetwork p<.01"
IMPORTANCE_P_THRESHOLD = 0.01
OUT_DIR = Path(os.path.join("/Users/xiaoqianxiao/projects/NARSAD/LSS/results", PIPELINE_NAME))
CHECKPOINT_DIR = OUT_DIR / "checkpoints"
LEGACY_CHECKPOINT_DIR = OUT_DIR / "checkpoint"
INTERMEDIATE_DIR = OUT_DIR / "intermediate"
FIGURE_DIR = OUT_DIR / "notebook_figures_pLess0.01"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Pipeline:", PIPELINE_LABEL)
print("Importance-mask threshold:", IMPORTANCE_P_THRESHOLD)
print("OUT_DIR:", OUT_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("LEGACY_CHECKPOINT_DIR:", LEGACY_CHECKPOINT_DIR)
print("INTERMEDIATE_DIR:", INTERMEDIATE_DIR)
print("Notebook figures will be saved to:", FIGURE_DIR)
_stage19_probe = CHECKPOINT_DIR / "cell_16_opening_test.joblib"
print("Stage 19 checkpoint:", _stage19_probe, "exists=", _stage19_probe.exists())


## Helper Functions


In [ ]:
def _existing(paths):
    for path in paths:
        p = Path(path)
        if p.exists():
            return p
    return None


def load_joblib(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        obj = joblib.load(path)
        plt.close("all")
        return obj
    except Exception as exc:
        print(f"Could not load {path}: {exc}")
        plt.close("all")
        return default


def load_first(paths, default=None):
    for path in paths:
        obj = load_joblib(path, default=None)
        if obj is not None:
            print(f"Loaded: {path}")
            return obj
    return default


def coerce_result(obj):
    if obj is None:
        return {}
    if isinstance(obj, dict):
        for key in ["results", "result", "payload", "data"]:
            if key in obj and isinstance(obj[key], dict):
                merged = dict(obj[key])
                for k, v in obj.items():
                    if k not in merged:
                        merged[k] = v
                return merged
        return obj
    return {"value": obj}


def first_present(mapping, keys):
    if mapping is None:
        return None
    for key in keys:
        if isinstance(mapping, dict) and key in mapping and mapping[key] is not None:
            return mapping[key]
    return None


def flatten_result_dict(d, prefix=""):
    rows = []
    if d is None:
        return pd.DataFrame(columns=["key", "value"])
    if not isinstance(d, dict):
        return pd.DataFrame([{"key": prefix or "value", "value": repr(d)}])
    for key, value in d.items():
        name = f"{prefix}.{key}" if prefix else str(key)
        if isinstance(value, dict):
            rows.extend(flatten_result_dict(value, name).to_dict("records"))
        elif isinstance(value, (str, int, float, bool, np.integer, np.floating)) or value is None:
            rows.append({"key": name, "value": value})
        elif isinstance(value, pd.DataFrame):
            rows.append({"key": name, "value": f"DataFrame shape={value.shape}"})
        elif isinstance(value, np.ndarray):
            rows.append({"key": name, "value": f"ndarray shape={value.shape}, dtype={value.dtype}"})
        elif isinstance(value, (list, tuple)):
            rows.append({"key": name, "value": f"{type(value).__name__} len={len(value)}"})
        else:
            rows.append({"key": name, "value": type(value).__name__})
    return pd.DataFrame(rows)


def display_df(df, rows=30):
    if df is None:
        display(Markdown("_No table available._"))
        return
    if isinstance(df, dict):
        df = flatten_result_dict(df)
    if not isinstance(df, pd.DataFrame):
        display(pd.DataFrame({"value": [repr(df)]}))
        return
    if df.empty:
        display(Markdown("_Table is empty._"))
        return
    display(df.head(rows))


def find_figures(*keywords):
    roots = [OUT_DIR, CHECKPOINT_DIR, LEGACY_CHECKPOINT_DIR, INTERMEDIATE_DIR]
    figs = []
    key_l = [k.lower() for k in keywords if k]
    for root in roots:
        if not root.exists():
            continue
        for ext in ("*.png", "*.jpg", "*.jpeg", "*.svg"):
            for path in root.rglob(ext):
                name = path.name.lower()
                if not key_l or any(k in name for k in key_l):
                    figs.append(path)
    return sorted(set(figs))


def show_figures_by_keywords(title, *keywords, max_figs=12):
    figs = find_figures(*keywords)
    display(Markdown(f"**Saved figures: {title}**"))
    if not figs:
        display(Markdown("_No saved figures matched these keywords._"))
        return []
    for path in figs[:max_figs]:
        display(Markdown(f"`{path}`"))
        if path.suffix.lower() == ".svg":
            display(Markdown(f"![{path.name}]({path})"))
        else:
            display(Image(filename=str(path)))
    if len(figs) > max_figs:
        display(Markdown(f"_Showing {max_figs} of {len(figs)} matched figures._"))
    return figs


def save_current_fig(name):
    path = FIGURE_DIR / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    print(f"Saved {path}")


def plot_matrix(mat, title, labels=None, ax=None, cmap="vlag", center=0):
    arr = np.asarray(mat)
    if arr.ndim == 3:
        arr = np.nanmean(arr, axis=0)
    if arr.ndim != 2:
        raise ValueError(f"Expected 2D/3D matrix, got shape {arr.shape}")
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(arr, ax=ax, cmap=cmap, center=center, square=True, annot=True, fmt=".3g", xticklabels=labels, yticklabels=labels, cbar_kws={"shrink": 0.75})
    ax.set_title(title)
    return ax


def maybe_plot_numeric_bars(df, title, max_cols=12):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        return False
    numeric = df.select_dtypes(include=[np.number])
    if numeric.empty:
        return False
    means = numeric.mean(numeric_only=True).sort_values(key=lambda s: s.abs(), ascending=False).head(max_cols)
    fig, ax = plt.subplots(figsize=(max(7, len(means) * 0.65), 4))
    sns.barplot(x=means.index, y=means.values, ax=ax, color="#4C78A8")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    ax.set_ylabel("Mean")
    save_current_fig(title.lower().replace(" ", "_").replace("/", "_") + ".png")
    plt.show()
    return True


def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    out = np.full(pvals.shape, np.nan, dtype=float)
    valid = np.isfinite(pvals)
    p = pvals[valid]
    if p.size == 0:
        return out
    order = np.argsort(p)
    ranked = p[order] * p.size / np.arange(1, p.size + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    ranked = np.clip(ranked, 0, 1)
    q = np.empty_like(ranked)
    q[order] = ranked
    out[valid] = q
    return out


def welch_by_trial(df, value_col, group_col="Group", trial_col="trial"):
    if df is None or not isinstance(df, pd.DataFrame):
        return pd.DataFrame()
    needed = {value_col, group_col, trial_col}
    if not needed.issubset(df.columns):
        return pd.DataFrame()
    rows = []
    for trial, sub in df.groupby(trial_col):
        sad = pd.to_numeric(sub.loc[sub[group_col].astype(str).str.upper().eq("SAD"), value_col], errors="coerce").dropna()
        hc = pd.to_numeric(sub.loc[sub[group_col].astype(str).str.upper().eq("HC"), value_col], errors="coerce").dropna()
        if len(sad) >= 2 and len(hc) >= 2:
            t, p = stats.ttest_ind(sad, hc, equal_var=False, nan_policy="omit")
            pooled = math.sqrt(((sad.std(ddof=1) ** 2) + (hc.std(ddof=1) ** 2)) / 2) if len(sad) > 1 and len(hc) > 1 else np.nan
            d = (sad.mean() - hc.mean()) / pooled if pooled and np.isfinite(pooled) and pooled > 0 else np.nan
        else:
            t, p, d = np.nan, np.nan, np.nan
        rows.append({"trial": trial, "n_sad": len(sad), "n_hc": len(hc), "mean_sad": sad.mean() if len(sad) else np.nan, "mean_hc": hc.mean() if len(hc) else np.nan, "t": t, "p": p, "cohens_d": d})
    out = pd.DataFrame(rows).sort_values("trial") if rows else pd.DataFrame()
    if not out.empty:
        out["p_fdr_bh"] = bh_fdr(out["p"].to_numpy())
    return out


def extract_first_dataframe(result, preferred=()):
    if not isinstance(result, dict):
        return None
    for key in preferred:
        val = result.get(key)
        if isinstance(val, pd.DataFrame):
            return val
    for val in result.values():
        if isinstance(val, pd.DataFrame):
            return val
    return None


def plot_corr_table(df, title):
    if df is None or not isinstance(df, pd.DataFrame) or df.empty:
        display(Markdown("_No correlation table available._"))
        return
    display_df(df, rows=40)
    possible_r = [c for c in df.columns if c.lower() in {"r", "rho", "corr", "correlation", "estimate"}]
    x_cols = [c for c in df.columns if any(s in c.lower() for s in ["neural", "metric", "index", "predictor"])]
    y_cols = [c for c in df.columns if any(s in c.lower() for s in ["clinical", "score", "outcome", "measure"])]
    if possible_r and x_cols and y_cols:
        try:
            piv = df.pivot_table(index=y_cols[0], columns=x_cols[0], values=possible_r[0], aggfunc="mean")
            fig, ax = plt.subplots(figsize=(max(7, 0.45 * piv.shape[1]), max(4, 0.35 * piv.shape[0])))
            sns.heatmap(piv, cmap="vlag", center=0, ax=ax, cbar_kws={"label": possible_r[0]})
            ax.set_title(title)
            save_current_fig(title.lower().replace(" ", "_").replace("/", "_") + ".png")
            plt.show()
        except Exception as exc:
            print(f"Could not make correlation heatmap: {exc}")


NEURAL_MEASURE_LABELS = {
    # Analysis 1.4 / 2.3 decision-boundary metrics
    "entropy": "Uncertainty (entropy)",
    "kurtosis": "Sharpness (kurtosis)",
    "variance": "Decision-probability variance",
    "boundary_separation": "Boundary separation: P(CSR|CSR) - P(CSR|CSS)",
    "decision_margin_css": "CSS decision margin: |P(CSR|CSS) - 0.5|",
    "decision_margin_all": "Overall decision margin: |P(CSR) - 0.5|",
    "p_csr_css": "Threat-like safety: P(CSR|CSS)",
    "p_csr_csr": "Threat evidence: P(CSR|CSR)",
    "Entropy": "Uncertainty (entropy)",
    "Kurtosis": "Sharpness (kurtosis)",
    "Variance": "Decision-probability variance",
    "Boundary Separation": "Boundary separation: P(CSR|CSR) - P(CSR|CSS)",
    "CSS Decision Margin": "CSS decision margin: |P(CSR|CSS) - 0.5|",
    "Overall Decision Margin": "Overall decision margin: |P(CSR) - 0.5|",
    "P(CSR) CSS": "Threat-like safety: P(CSR|CSS)",
    "P(CSR) CSR": "Threat evidence: P(CSR|CSR)",
    "P_CSR_CSS": "Threat-like safety: P(CSR|CSS)",
    "P_CSR_CSR": "Threat evidence: P(CSR|CSR)",
    "Boundary_Separation": "Boundary separation: P(CSR|CSR) - P(CSR|CSS)",
    "Decision_Margin_CSS": "CSS decision margin: |P(CSR|CSS) - 0.5|",
    "Decision_Margin_All": "Overall decision margin: |P(CSR) - 0.5|",
    # Master neural-clinical names
    "Neural_Dist_Threat_Safety": "Threat-safety representational distance",
    "Neural_Dist_Safety_Background": "Safety-background representational distance",
    "Neural_Safety_Mean": "Safety-restoration distance",
    "Neural_Threat_Mean": "Threat-discrimination distance",
    "Neural_Safety_Mean_PV": "Safety-restoration distance (per voxel)",
    "Neural_Threat_Mean_PV": "Threat-discrimination distance (per voxel)",
    "Neural_SCR_Safety_Coupling": "SCR-neural safety coupling",
    "Neural_Uncertainty_Entropy": "Uncertainty (entropy)",
    "Neural_Sharpness_Kurtosis": "Sharpness (kurtosis)",
    "Neural_Decision_Margin_CSS": "CSS decision margin: |P(CSR|CSS) - 0.5|",
    "Neural_ThreatLike_Safety": "Threat-like safety: P(CSR|CSS)",
    "Neural_Threat_Evidence_CSR": "Threat evidence: P(CSR|CSR)",
    "Neural_Boundary_Separation": "Boundary separation: P(CSR|CSR) - P(CSR|CSS)",
}

CLINICAL_MEASURE_LABELS = {
    "lsas_total": "LSAS total",
    "lsas_fear": "LSAS fear",
    "lsas_avoid": "LSAS avoidance",
    "dass_anxiety": "DASS anxiety",
    "dass_stress": "DASS stress",
    "ecr_total": "ECR total",
    "demo_age": "Age",
}

def measure_label(name):
    if name is None:
        return ""
    s = str(name)
    base = s[:-2] if s.endswith("_z") else s
    label = NEURAL_MEASURE_LABELS.get(base) or CLINICAL_MEASURE_LABELS.get(base)
    if label is None:
        label = base.replace("Neural_", "").replace("_", " ").strip().title()
    return f"{label} (z)" if s.endswith("_z") else label

def add_measure_labels(df, columns=("metric", "measure", "Neural", "Clinical", "variable", "Neural_z", "Clinical_z")):
    if not isinstance(df, pd.DataFrame) or df.empty:
        return df
    out = df.copy()
    for col in columns:
        if col in out.columns and f"{col}_label" not in out.columns:
            out[f"{col}_label"] = out[col].map(measure_label)
    return out


def memoryfear_roi_display_order(existing_rois=None):
    """Display MemoryFearNetwork ROIs as L/R pairs: memory ROIs first, then fear-network ROIs."""
    memory_pairs = ["hippocampus", "AG", "SMG", "IFG", "MFG", "SFG", "Precuneus", "VVC"]
    fear_pairs = ["acc", "amygdala", "insula", "vmpfc"]
    order = []
    for roi_base in memory_pairs + fear_pairs:
        order.extend([f"left_{roi_base}", f"right_{roi_base}"])
    if existing_rois is not None:
        existing = list(existing_rois)
        order = [roi for roi in order if roi in set(existing)]
        order.extend([roi for roi in existing if roi not in order])
    return order

def roi_display_order_for_pipeline(existing_rois=None):
    if PIPELINE_NAME == "MemoryFearNetwork":
        return memoryfear_roi_display_order(existing_rois)
    base_order = ["left_hippocampus", "right_hippocampus", "left_insula", "right_insula", "left_acc", "right_acc", "left_vmpfc", "right_vmpfc", "left_amygdala", "right_amygdala"]
    if existing_rois is not None:
        existing = list(existing_rois)
        base_order = [roi for roi in base_order if roi in set(existing)]
        base_order.extend([roi for roi in existing if roi not in base_order])
    return base_order


def roi_display_label(roi):
    text = str(roi)
    hemi = ""
    if text.startswith("left_"):
        hemi = "L "
        text = text[5:]
    elif text.startswith("right_"):
        hemi = "R "
        text = text[6:]
    return hemi + text.replace("_", " ")


## Output Manifest


In [ ]:
# Load saved stage/checkpoint payloads. Missing files are expected if a stage has not run yet.
paths = {
    "stage03_data": [CHECKPOINT_DIR / "cell_03.joblib", LEGACY_CHECKPOINT_DIR / "cell_03.joblib"],
    "stage06_results_11": [CHECKPOINT_DIR / "results_analysis_11.joblib", CHECKPOINT_DIR / "results_11.joblib", CHECKPOINT_DIR / "cell_06.joblib", LEGACY_CHECKPOINT_DIR / "results_analysis_11.joblib", LEGACY_CHECKPOINT_DIR / "results_11.joblib", LEGACY_CHECKPOINT_DIR / "cell_06.joblib"],
    "stage10_haufe": [INTERMEDIATE_DIR / "stage10_haufe_maps.joblib", CHECKPOINT_DIR / "cell_10.joblib", LEGACY_CHECKPOINT_DIR / "cell_10.joblib"],
    "stage11_importance": [INTERMEDIATE_DIR / "stage11_importance_masks.joblib", CHECKPOINT_DIR / "cell_11.joblib"],
    "stage12_topology": [CHECKPOINT_DIR / "analysis_12_topology.joblib", CHECKPOINT_DIR / "results_analysis_12.joblib", INTERMEDIATE_DIR / "stage12_topology_stats.joblib", INTERMEDIATE_DIR / "stage12_StaticRepresentationalTopology.joblib", INTERMEDIATE_DIR / "stage12_topology.joblib", CHECKPOINT_DIR / "results_12.joblib", CHECKPOINT_DIR / "cell_12.joblib", CHECKPOINT_DIR / "cell_10.joblib"],
    "stage13_drift": [CHECKPOINT_DIR / "results_analysis_13.joblib", INTERMEDIATE_DIR / "stage13_drift.joblib", CHECKPOINT_DIR / "cell_13.joblib"],
    "stage14_trajectories": [CHECKPOINT_DIR / "results_analysis_13_trajectories.joblib", INTERMEDIATE_DIR / "stage14_trajectories.joblib", CHECKPOINT_DIR / "cell_14.joblib"],
    "stage15_decision": [CHECKPOINT_DIR / "results_analysis_14.joblib", INTERMEDIATE_DIR / "stage15_decision_boundary.joblib", CHECKPOINT_DIR / "cell_15.joblib"],
    "stage16_safety_threat": [CHECKPOINT_DIR / "cell_24.joblib", CHECKPOINT_DIR / "cell_14.joblib", CHECKPOINT_DIR / "results_analysis_21.joblib", INTERMEDIATE_DIR / "stage16_safety_threat.joblib"],
    "stage18_drift_efficiency": [CHECKPOINT_DIR / "cell_24.joblib", CHECKPOINT_DIR / "cell_15.joblib", CHECKPOINT_DIR / "results_analysis_22.joblib", INTERMEDIATE_DIR / "stage18_drift_efficiency.joblib"],
    "stage19_prob_opening": [CHECKPOINT_DIR / "cell_16_opening_test.joblib", CHECKPOINT_DIR / "cell_16.joblib", CHECKPOINT_DIR / "results_analysis_23.joblib", INTERMEDIATE_DIR / "stage19_probabilistic_opening.joblib"],
    "stage20_realignment": [CHECKPOINT_DIR / "cell_17_realignment.joblib", CHECKPOINT_DIR / "results_analysis_24.joblib", INTERMEDIATE_DIR / "stage20_spatial_realignment.joblib", CHECKPOINT_DIR / "cell_20.joblib"],
    "stage21_reverse": [CHECKPOINT_DIR / "cell_18_reverse_cross_decoding.joblib", CHECKPOINT_DIR / "cell_18_realignment.joblib", CHECKPOINT_DIR / "results_analysis_25.joblib", INTERMEDIATE_DIR / "stage21_reverse_cross_decoding.joblib", CHECKPOINT_DIR / "cell_21.joblib"],
    "stage23_clinical_scores": [INTERMEDIATE_DIR / "stage23_clinical_scores.joblib", CHECKPOINT_DIR / "cell_23.joblib"],
    "stage24_neural_indices": [INTERMEDIATE_DIR / "stage24_neural_clinical_indices.joblib", CHECKPOINT_DIR / "cell_24.joblib"],
    "stage26_master_merge": [CHECKPOINT_DIR / "cell_26.joblib", INTERMEDIATE_DIR / "stage26_master_neural_clinical.joblib"],
    "stage27_pearson": [INTERMEDIATE_DIR / "stage27_neural_clinical_pearson.joblib", CHECKPOINT_DIR / "cell_27.joblib"],
    "stage28_partial": [INTERMEDIATE_DIR / "stage28_neural_clinical_partial.joblib", CHECKPOINT_DIR / "cell_28.joblib"],
    "stage29_zscore_outlier": [INTERMEDIATE_DIR / "stage29_zscore_outlier.joblib", CHECKPOINT_DIR / "cell_29.joblib"],
    "stage30_ols": [INTERMEDIATE_DIR / "stage30_z_ols_regression.joblib", CHECKPOINT_DIR / "cell_30.joblib"],
}
loaded = {name: load_first(candidates, default=None) for name, candidates in paths.items()}
results = {name: coerce_result(payload) for name, payload in loaded.items()}


def apply_importance_p_threshold(stage11_payload, alpha=0.01):
    if not isinstance(stage11_payload, dict):
        return stage11_payload
    out = dict(stage11_payload)
    pvals = None
    for key in ["p_values_permutated", "p_values", "pvals"]:
        if isinstance(out.get(key), dict):
            pvals = out[key]
            break
    if not isinstance(pvals, dict):
        print(f"[p<{alpha:g} mask override] No per-group p-value dict found; keeping saved importance masks.")
        return out
    new_masks = {}
    counts = {}
    counts_p05 = {}
    for group, pv in pvals.items():
        arr = np.asarray(pv, dtype=float)
        mask = np.isfinite(arr) & (arr < alpha)
        new_masks[group] = mask
        counts[group] = int(mask.sum())
        counts_p05[group] = int((np.isfinite(arr) & (arr < 0.05)).sum())
    out["importance_mask_permutated"] = new_masks
    out["importance_mask_threshold"] = alpha
    out["importance_mask_threshold_note"] = f"Visualization override: permutation p < {alpha:g}"
    out["importance_mask_counts_p05_uncorrected"] = counts_p05
    print(f"[p<{alpha:g} mask override] stage11 importance counts:", counts)
    print("[p<0.05 check] stage11 uncorrected p<0.05 counts:", counts_p05)
    for group, n05 in counts_p05.items():
        if n05 == 0:
            print(f"[p<0.05 check] {group}: no voxels survive uncorrected p < 0.05.")
    return out
results["stage11_importance"] = apply_importance_p_threshold(results.get("stage11_importance", {}), IMPORTANCE_P_THRESHOLD)
manifest = []
for name, candidates in paths.items():
    existing = _existing(candidates)
    manifest.append({"stage": name, "found": existing is not None, "path": str(existing) if existing else ""})
display_df(pd.DataFrame(manifest), rows=50)


## 1. Analysis 1.1: Neural Dissociation Self-Decoding
This section summarizes self-decoding accuracy with null distribution, functional specificity, and spatial specificity where those outputs were saved by the script.


In [ ]:
r11_payload = results.get("stage06_results_11", {})
r11 = r11_payload.get("results_11", r11_payload) if isinstance(r11_payload, dict) else {}
display_df(flatten_result_dict(r11), rows=60)

# Recreate the four-panel Analysis 1.1 figure: self-decoding null distributions, functional specificity, and spatial specificity.
def _lookup_many(*keys):
    for key in keys:
        if isinstance(r11, dict) and key in r11 and r11[key] is not None:
            return r11[key]
        if isinstance(r11_payload, dict) and key in r11_payload and r11_payload[key] is not None:
            return r11_payload[key]
    return None


def _scalar_or_nan(value):
    try:
        arr = np.asarray(value, dtype=float)
        if arr.size == 1:
            return float(arr.ravel()[0])
    except Exception:
        pass
    return np.nan


def _format_p(p):
    p = _scalar_or_nan(p)
    if np.isfinite(p):
        return f"p = {p:.4f}"
    return "p = n/a"


def _plot_self_decoding(ax, null_dist, obs, p_val, title):
    obs = _scalar_or_nan(obs)
    null_arr = np.asarray(null_dist, dtype=float).ravel() if null_dist is not None else np.array([])
    null_arr = null_arr[np.isfinite(null_arr)]
    if null_arr.size:
        sns.histplot(null_arr, stat="density", bins=35, color="lightgray", edgecolor="black", alpha=0.85, ax=ax, label="Null Dist")
        try:
            sns.kdeplot(null_arr, ax=ax, color="gray", linewidth=2.5)
        except Exception:
            pass
        thresh = np.nanpercentile(null_arr, 95)
        ax.axvline(thresh, color="blue", linestyle="--", linewidth=2, label="95% null")
    else:
        ax.text(0.5, 0.5, "Null distribution not found", ha="center", va="center", transform=ax.transAxes)
    if np.isfinite(obs):
        ax.axvline(obs, color="red", linewidth=2.5, label=f"Obs: {obs:.2f}")
    ax.set_title(f"{title} (CV Acc: {obs:.2f})\n({_format_p(p_val)})" if np.isfinite(obs) else f"{title}\n({_format_p(p_val)})")
    ax.set_xlabel("Forced-choice accuracy")
    ax.set_ylabel("Density")
    ax.legend(frameon=True, fontsize=10)

acc_sad = _lookup_many("acc_sad_cv", "accuracy_sad", "sad_accuracy")
acc_hc = _lookup_many("acc_hc_cv", "accuracy_hc", "hc_accuracy")
p_sad = _lookup_many("p_sad", "p_acc_sad", "sad_p")
p_hc = _lookup_many("p_hc", "p_acc_hc", "hc_p")
perm_sad = _lookup_many("perm_dist_sad", "perm_acc_sad", "null_dist_sad", "sad_null")
perm_hc = _lookup_many("perm_dist_hc", "perm_acc_hc", "null_dist_hc", "hc_null")
func_matrix = _lookup_many("func_matrix", "functional_specificity", "functional_matrix")
func_pvals = _lookup_many("p_func_pvals", "func_pvals", "functional_pvals")
sim_spatial = _lookup_many("sim_spatial", "obs_sim", "patterm_similarity", "pattern_similarity")
p_sim = _lookup_many("p_sim", "p_sim_spatial", "p_spatial")
spatial_matrix = _lookup_many("spatial_matrix")
spatial_pvals = _lookup_many("spatial_pvals")

if func_matrix is None:
    func_matrix = np.array([[_scalar_or_nan(acc_sad), np.nan], [np.nan, _scalar_or_nan(acc_hc)]], dtype=float)
else:
    func_matrix = np.asarray(func_matrix, dtype=float)
if func_pvals is None:
    func_pvals = np.array([[_scalar_or_nan(p_sad), np.nan], [np.nan, _scalar_or_nan(p_hc)]], dtype=float)
else:
    func_pvals = np.asarray(func_pvals, dtype=float)
if spatial_matrix is None:
    sim = _scalar_or_nan(sim_spatial)
    spatial_matrix = np.array([[1.0, sim], [sim, 1.0]], dtype=float)
else:
    spatial_matrix = np.asarray(spatial_matrix, dtype=float)
if spatial_pvals is None:
    ps = _scalar_or_nan(p_sim)
    spatial_pvals = np.array([[0.0, ps], [ps, 0.0]], dtype=float)
else:
    spatial_pvals = np.asarray(spatial_pvals, dtype=float)

if any(x is not None for x in [perm_sad, perm_hc, func_matrix, spatial_matrix]):
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 2, height_ratios=[1.0, 1.25], hspace=0.38, wspace=0.28)
    _plot_self_decoding(fig.add_subplot(gs[0, 0]), perm_sad, acc_sad, p_sad, "SAD Self-Decoding")
    _plot_self_decoding(fig.add_subplot(gs[0, 1]), perm_hc, acc_hc, p_hc, "HC Self-Decoding")
    annot_func = np.empty(func_matrix.shape, dtype=object)
    for i in range(func_matrix.shape[0]):
        for j in range(func_matrix.shape[1]):
            val = func_matrix[i, j]
            pv = func_pvals[i, j] if func_pvals.shape == func_matrix.shape else np.nan
            star = "*" if np.isfinite(pv) and pv < 0.05 else ""
            annot_func[i, j] = f"{val:.3f}\n({star})" if np.isfinite(val) else "n/a"
    ax3 = fig.add_subplot(gs[1, 0])
    sns.heatmap(func_matrix, annot=annot_func, fmt="", cmap="RdBu_r", center=0.5, vmin=0.3, vmax=0.9, cbar=True, xticklabels=["Test SAD", "Test HC"], yticklabels=["Train SAD", "Train HC"], ax=ax3)
    ax3.set_title("Functional Specificity\n(Forced-Choice Accuracy)")
    annot_spatial = np.empty(spatial_matrix.shape, dtype=object)
    for i in range(spatial_matrix.shape[0]):
        for j in range(spatial_matrix.shape[1]):
            val = spatial_matrix[i, j]
            pv = spatial_pvals[i, j] if spatial_pvals.shape == spatial_matrix.shape else np.nan
            star = "*" if i != j and np.isfinite(pv) and pv < 0.05 else ""
            annot_spatial[i, j] = f"{val:.3f}\n{star}" if np.isfinite(val) else "n/a"
    ax4 = fig.add_subplot(gs[1, 1])
    sns.heatmap(spatial_matrix, annot=annot_spatial, fmt="", cmap="RdBu_r", center=0, vmin=-1, vmax=1, cbar=True, xticklabels=["SAD Map", "HC Map"], yticklabels=["SAD Map", "HC Map"], ax=ax4)
    ax4.set_title("Spatial Specificity")
    save_current_fig("section01_neural_dissociation_four_panel_pLess0.01.png")
    plt.show()
else:
    display(Markdown("_Analysis 1.1 four-panel ingredients were not found in the saved payload._"))
    show_figures_by_keywords("Saved Analysis 1.1 fallback", "analysis_11", "results_11", "neural_dissociation", "functional_specificity", "spatial_specificity", max_figs=4)

for key in ["acc_summary", "accuracy_summary", "df_accuracy", "df_acc", "functional_specificity", "spatial_specificity", "null_distribution"]:
    val = r11.get(key) if isinstance(r11, dict) else None
    if isinstance(val, pd.DataFrame):
        display(Markdown(f"**{key}**"))
        display_df(val, rows=30)
    elif isinstance(val, np.ndarray) and val.ndim in (1, 2):
        fig, ax = plt.subplots(figsize=(6, 4))
        if val.ndim == 1:
            ax.hist(val[np.isfinite(val)], bins=30, color="#4C78A8", alpha=0.8)
            ax.set_title(key)
        else:
            plot_matrix(val, key, ax=ax)
        save_current_fig(f"section01_{key}.png")
        plt.show()


## 2. Haufe / Z-Scored Maps (p < .01)
Glass-brain figures are reconstructed from permutation-cache p-values using the strict `IMPORTANCE_P_THRESHOLD = 0.01`. This variant does not fall back to saved p < .05 Haufe figures.


In [ ]:
r10 = results.get("stage10_haufe", {})
r03 = results.get("stage03_data", {})
r11_payload = results.get("stage06_results_11", {})
r11_for_maps = r11_payload.get("results_11", r11_payload) if isinstance(r11_payload, dict) else {}
display_df(flatten_result_dict(r10), rows=60)

# FearNetworkAll/MemoryFearNetwork-style glass-brain reconstruction for ROI permutation maps.
def _nested_get(mapping, keys):
    if not isinstance(mapping, dict):
        return None
    for key in keys:
        if key in mapping and mapping[key] is not None:
            return mapping[key]
    for value in mapping.values():
        if isinstance(value, dict):
            found = _nested_get(value, keys)
            if found is not None:
                return found
    return None


def _pipeline_roi_candidates():
    if PIPELINE_NAME == "FearNetwork":
        roots = [
            Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/Gillian_anatomically_constrained"),
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/NARSAD_uni/Gillian_NARSAD"),
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/FearNetwork"),
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/FearNetwork"),
        ]
    elif PIPELINE_NAME == "MemoryFearNetwork":
        roots = [
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs/MemoryFearNetwork"),
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI/MemoryFearNetwork"),
            Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
        ]
    else:
        roots = [
            Path("/Users/xiaoqianxiao/projects/NARSAD/ROI") / PIPELINE_NAME,
            Path("/Users/xiaoqianxiao/tool/parcellation/ROIs") / PIPELINE_NAME,
            Path("/Users/xiaoqianxiao/tool/parcellation/Gillian_anatomically_constrained"),
        ]
    env_name = "MEMORY_FEAR_ROI_DIR" if PIPELINE_NAME == "MemoryFearNetwork" else "FEAR_ROI_DIR"
    env_path = os.environ.get(env_name) or os.environ.get("ROI_DIR")
    if env_path:
        roots.insert(0, Path(env_path))
    return [p for p in roots if p.exists()]


def _default_roi_order():
    rois = _nested_get(r03, ["TARGET_ROIS", "ROI_ORDER", "roi_order", "target_rois"])
    if rois is not None:
        return list(rois)
    if PIPELINE_NAME == "MemoryFearNetwork":
        return ["left_acc", "left_amygdala", "left_hippocampus", "left_insula", "left_vmpfc", "left_VVC", "left_AG", "left_SMG", "left_IFG", "left_MFG", "left_SFG", "left_Precuneus", "right_acc", "right_amygdala", "right_hippocampus", "right_insula", "right_vmpfc", "right_VVC", "right_AG", "right_SMG", "right_IFG", "right_MFG", "right_SFG", "right_Precuneus"]
    return ["left_acc", "left_amygdala", "left_hippocampus", "left_insula", "left_vmpfc", "right_acc", "right_amygdala", "right_hippocampus", "right_insula", "right_vmpfc"]


def _find_roi_dir(roi_order):
    for roi_dir in _pipeline_roi_candidates():
        if all(list(roi_dir.glob(f"*{roi}*.nii*")) for roi in roi_order):
            return roi_dir
    return None


def _roi_chunk_lengths(roi_order, roi_dir):
    try:
        import nibabel as nib
    except Exception:
        return None, None, "nibabel unavailable"
    chunks = []
    ref_img = None
    for roi in roi_order:
        files = sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))
        if not files:
            return None, None, f"missing mask for {roi} in {roi_dir}"
        img = nib.load(str(files[0]))
        if ref_img is None:
            ref_img = img
        chunks.append(int(np.sum(img.get_fdata() > 0)))
    return chunks, ref_img, None


def _bh_mask(pvals, alpha=None):
    if alpha is None:
        alpha = IMPORTANCE_P_THRESHOLD
    p = np.asarray(pvals, dtype=float).ravel()
    valid = np.isfinite(p)
    out = np.zeros(p.shape, dtype=bool)
    if valid.sum() == 0:
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    thresh = alpha * np.arange(1, len(ranked) + 1) / len(ranked)
    passed = ranked <= thresh
    if passed.any():
        cutoff = ranked[np.where(passed)[0].max()]
        out[valid] = pv <= cutoff
    return out


def _roi_fdr_mask(pvals, roi_order, roi_dir, alpha=None):
    if alpha is None:
        alpha = IMPORTANCE_P_THRESHOLD
    chunks, _, err = _roi_chunk_lengths(roi_order, roi_dir)
    p = np.asarray(pvals, dtype=float).ravel()
    if chunks is None or sum(chunks) != p.size:
        return _bh_mask(p, alpha=alpha), f"Global FDR p<{IMPORTANCE_P_THRESHOLD:g}; ROI masks not found"
    mask = np.zeros(p.shape, dtype=bool)
    cursor = 0
    for n in chunks:
        mask[cursor: cursor + n] = _bh_mask(p[cursor: cursor + n], alpha=alpha)
        cursor += n
    return mask, "ROI-FDR"


def _load_perm_cache(group):
    candidates = [
        CHECKPOINT_DIR / f"perm_results_{group}_memoryfear_network_2way.joblib",
        LEGACY_CHECKPOINT_DIR / f"perm_results_{group}_memoryfear_network_2way.joblib",
        OUT_DIR / f"perm_results_{group}_memoryfear_network_2way.joblib",
        CHECKPOINT_DIR / f"perm_results_{group}_fear_network_2way.joblib",
        LEGACY_CHECKPOINT_DIR / f"perm_results_{group}_fear_network_2way.joblib",
        OUT_DIR / f"perm_results_{group}_fear_network_2way.joblib",
        CHECKPOINT_DIR / f"perm_results_{group}_wholeBrain_voxelwise.joblib",
        LEGACY_CHECKPOINT_DIR / f"perm_results_{group}_wholeBrain_voxelwise.joblib",
        OUT_DIR / f"perm_results_{group}_wholeBrain_voxelwise.joblib",
    ]
    for path in candidates:
        obj = load_joblib(path, default=None)
        if isinstance(obj, dict) and {"obs_weights", "null_weights", "p_values_raw"}.issubset(obj.keys()):
            return obj, path
    return None, None


def _z_from_cache(cache):
    obs = np.asarray(cache["obs_weights"], dtype=float).ravel()
    null = np.asarray(cache["null_weights"], dtype=float)
    z = (obs - np.nanmean(null, axis=0)) / (np.nanstd(null, axis=0) + 1e-12)
    z = np.asarray(z, dtype=float).ravel()
    # The source notebooks flip CSR-first models so CSS/threat-positive evidence is red.
    if PIPELINE_NAME in {"FearNetwork", "MemoryFearNetwork"}:
        z = -z
    return z



def _roi_selection_stats(sig_mask, roi_order, roi_dir, group, mode):
    chunks, _, err = _roi_chunk_lengths(roi_order, roi_dir)
    if chunks is None:
        return pd.DataFrame([{"Group": group, "Mode": mode, "ROI": "ROI stats unavailable", "ROI_voxels": np.nan, "Selected_voxels": int(np.sum(sig_mask)), "Percent_ROI": np.nan, "Note": err}])
    rows = []
    cursor = 0
    sig_arr = np.asarray(sig_mask, dtype=bool).ravel()
    for roi, n in zip(roi_order, chunks):
        roi_mask = sig_arr[cursor: cursor + n]
        selected = int(np.sum(roi_mask))
        rows.append({"Group": group, "Mode": mode, "ROI": roi, "ROI_voxels": int(n), "Selected_voxels": selected, "Percent_ROI": 100.0 * selected / n if n else np.nan, "Note": ""})
        cursor += n
    return pd.DataFrame(rows)

def _matched_perm_maps():
    roi_order = _default_roi_order()
    roi_dir = _find_roi_dir(roi_order)
    caches = {}
    for group in ["SAD", "HC"]:
        cache, path = _load_perm_cache(group)
        if cache is not None:
            caches[group] = {"cache": cache, "path": path, "z": _z_from_cache(cache), "p": np.asarray(cache["p_values_raw"], dtype=float).ravel()}
    if set(caches) != {"SAD", "HC"}:
        return {}, [f"Permutation caches found for groups: {sorted(caches)}"]
    fdr_masks = {}
    fdr_counts = {}
    fdr05_counts = {}
    mode0 = {}
    for group, payload in caches.items():
        if roi_dir is not None:
            fdr_masks[group], mode0[group] = _roi_fdr_mask(payload["p"], roi_order, roi_dir)
            mask05, _ = _roi_fdr_mask(payload["p"], roi_order, roi_dir, alpha=0.05)
        else:
            fdr_masks[group], mode0[group] = _bh_mask(payload["p"]), f"Global FDR p<{IMPORTANCE_P_THRESHOLD:g}; ROI masks not found"
            mask05 = _bh_mask(payload["p"], alpha=0.05)
        fdr_counts[group] = int(np.sum(fdr_masks[group]))
        fdr05_counts[group] = int(np.sum(mask05))
    # In this strict-threshold notebook, keep the exact p < .01/FDR masks.
    # Do not use matched-top-N or top-2% fallbacks, because those would change
    # the feature-space definition away from the requested p-value threshold.
    match_cfg = {"SAD": None, "HC": None}
    maps = {}
    messages = [f"ROI_DIR={roi_dir}", f"Initial ROI-FDR counts at p < {IMPORTANCE_P_THRESHOLD:g}: SAD={fdr_counts['SAD']}, HC={fdr_counts['HC']}", f"Reference ROI-FDR counts at p < 0.05: SAD={fdr05_counts['SAD']}, HC={fdr05_counts['HC']}"]
    for group, n05 in fdr05_counts.items():
        if n05 == 0:
            messages.append(f"{group}: no voxels survive ROI-FDR p < 0.05.")
    for group, payload in caches.items():
        z = payload["z"]
        cfg = match_cfg[group]
        if cfg is None:
            sig_mask = fdr_masks[group]
            mode = f"{mode0[group]} p<{IMPORTANCE_P_THRESHOLD:g}"
        else:
            sig_mask = fdr_masks[group]
            mode = f"{mode0[group]} p<{IMPORTANCE_P_THRESHOLD:g}"
        z_masked = z * sig_mask
        display_mask = sig_mask.copy()
        roi_stats = _roi_selection_stats(display_mask, roi_order, roi_dir, group, mode) if roi_dir is not None else pd.DataFrame()
        maps[group] = {"values": z_masked, "mode": mode, "path": payload["path"], "n": int(np.sum(display_mask)), "selection_n_before_direction_filter": int(np.sum(sig_mask)), "roi_order": roi_order, "roi_dir": roi_dir, "roi_stats": roi_stats}
    return maps, messages


def _as_img_like(obj):
    if obj is None:
        return None
    try:
        import nibabel as nib
        from nilearn import image
        if isinstance(obj, nib.spatialimages.SpatialImage):
            return obj
        if isinstance(obj, (str, Path)) and Path(obj).exists():
            return image.load_img(str(obj))
    except Exception:
        return None
    return None


def _reconstruct_roi_map(flat_data, roi_order, roi_dir):
    try:
        import nibabel as nib
    except Exception as exc:
        return None, f"nibabel unavailable: {exc}"
    chunks, ref_img, err = _roi_chunk_lengths(roi_order, roi_dir)
    if chunks is None:
        return None, err
    arr = np.asarray(flat_data, dtype=float).ravel()
    if sum(chunks) != arr.size:
        return None, f"ROI mask voxel count {sum(chunks)} does not match map length {arr.size}"
    final_vol = np.zeros(ref_img.shape, dtype=float)
    cursor = 0
    for roi, n in zip(roi_order, chunks):
        files = sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))
        mask_img = nib.load(str(files[0]))
        mask_data = mask_img.get_fdata() > 0
        final_vol[mask_data] = arr[cursor: cursor + n]
        cursor += n
    return nib.Nifti1Image(final_vol, ref_img.affine, ref_img.header), None


def _vector_to_img(values, map_info=None):
    try:
        import nibabel as nib
        from nilearn import masking
    except Exception as exc:
        return None, f"nilearn/nibabel unavailable: {exc}"
    img = _as_img_like(values)
    if img is not None:
        return img, None
    arr = np.asarray(values, dtype=float).ravel() if values is not None else np.array([])
    if arr.size == 0:
        return None, "map vector not found"
    if isinstance(map_info, dict) and map_info.get("roi_dir") is not None:
        img, err = _reconstruct_roi_map(arr, map_info["roi_order"], map_info["roi_dir"])
        if img is not None:
            return img, None
    mask_source = _nested_get(r10, ["mask_img", "brain_mask", "mask_ext", "roi_mask", "wholebrain_mask"])
    if mask_source is None:
        mask_source = _nested_get(r11_for_maps, ["mask_img", "brain_mask", "mask_ext", "roi_mask", "wholebrain_mask"])
    mask_img = _as_img_like(mask_source)
    if mask_img is not None:
        try:
            if int(mask_img.get_fdata().astype(bool).sum()) == arr.size:
                return masking.unmask(arr, mask_img), None
        except Exception as exc:
            return None, f"mask unmask failed: {exc}"
    return None, f"map has {arr.size} values but no compatible ROI masks or whole-brain mask were found"


def _plot_glass(group, map_info, ax=None):
    try:
        from nilearn import plotting
    except Exception as exc:
        return False, f"nilearn plotting unavailable: {exc}"
    img, err = _vector_to_img(map_info.get("values"), map_info)
    if img is None:
        return False, err
    data = np.asarray(img.get_fdata(), dtype=float)
    finite = data[np.isfinite(data)]
    finite_nonzero = finite[finite != 0]
    if finite_nonzero.size == 0:
        return False, "map image contains no non-zero finite values"
    vmax = float(np.nanpercentile(np.abs(finite_nonzero), 99))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = float(np.nanmax(np.abs(finite_nonzero)))
    threshold = float(np.nanmin(np.abs(finite_nonzero)))
    cmap = "RdBu_r"
    title = f"{group} {PIPELINE_LABEL}: {map_info.get('mode')}"
    plotting.plot_glass_brain(img, display_mode="lyrz", colorbar=True, threshold=threshold, vmax=vmax, cmap=cmap, plot_abs=False, black_bg=False, axes=ax, title=title)
    return True, map_info.get("mode")

matched_maps, match_messages = _matched_perm_maps()
if matched_maps:
    fig = plt.figure(figsize=(10, 8))
    axes = {"SAD": fig.add_subplot(2, 1, 1), "HC": fig.add_subplot(2, 1, 2)}
    messages = []
    rendered = []
    for group, ax in axes.items():
        ok, msg = _plot_glass(group, matched_maps[group], ax=ax)
        rendered.append(ok)
        if not ok:
            ax.axis("off")
            ax.text(0.5, 0.5, f"{group} glass brain could not be reconstructed\n{msg}", ha="center", va="center", wrap=True)
            messages.append(f"{group}: {msg}")
    save_current_fig("section02_memoryfearnetwork_roi_fdr_glass_brain_pLess0.01.png")
    plt.show()
    if messages:
        display(Markdown("; ".join(messages)))
    group_summary = pd.DataFrame([{ "group": g, "mode": v["mode"], "selected_voxels_shown": v["n"], "selected_before_direction_filter": v.get("selection_n_before_direction_filter", v["n"]), "cache": str(v["path"]), "roi_dir": str(v["roi_dir"]) } for g, v in matched_maps.items()])
    display_df(group_summary, rows=10)
    roi_plot_frames = []
    for group in ["SAD", "HC"]:
        roi_stats = matched_maps[group].get("roi_stats")
        mode = matched_maps[group].get("mode", "")
        print(f"\nResults for {group} ({mode}):")
        print(f"{'ROI Name':<25} | {'Voxels':<8} | {'% ROI'}")
        print("-" * 50)
        if isinstance(roi_stats, pd.DataFrame) and not roi_stats.empty:
            roi_print = roi_stats[roi_stats["Selected_voxels"] > 0].copy()
            roi_print_order = roi_display_order_for_pipeline(roi_print["ROI"].tolist())
            roi_print["ROI_order"] = pd.Categorical(roi_print["ROI"], categories=roi_print_order, ordered=True)
            roi_print = roi_print.sort_values("ROI_order")
            if roi_print.empty:
                print("No displayed significant voxels.")
            for _, row in roi_print.iterrows():
                print(f"{row['ROI']:<25} | {int(row['Selected_voxels']):<8} | {row['Percent_ROI']:.2f}%")
            roi_plot_frames.append(roi_stats.copy())
        else:
            print("ROI statistics unavailable.")
    if roi_plot_frames:
        roi_plot = pd.concat(roi_plot_frames, ignore_index=True)
        roi_plot = roi_plot[roi_plot["Selected_voxels"] > 0].copy()
        if not roi_plot.empty:
            roi_order_plot = roi_display_order_for_pipeline(roi_plot["ROI"].unique().tolist())
            roi_plot["ROI_label"] = roi_plot["ROI"].map(roi_display_label)
            roi_order_labels = [roi_display_label(roi) for roi in roi_order_plot]
            fig_roi, ax_roi = plt.subplots(figsize=(9, max(6, 0.42 * len(roi_order_labels))))
            sns.barplot(data=roi_plot, y="ROI_label", x="Percent_ROI", hue="Group", order=roi_order_labels, palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=ax_roi)
            for container in ax_roi.containers:
                labels = []
                for bar in container:
                    width = bar.get_width()
                    labels.append(f"{width:.1f}%" if np.isfinite(width) and width > 0 else "")
                ax_roi.bar_label(container, labels=labels, padding=3, fontsize=8)
            ax_roi.set_xlim(0, max(roi_plot["Percent_ROI"].max() * 1.35, 1))
            ax_roi.set_xlabel("Selected voxels (% of ROI)")
            ax_roi.set_ylabel("")
            ax_roi.set_title("ROI distribution of selected significant voxels")
            ax_roi.tick_params(axis='y', labelsize=9)
            save_current_fig("section02_roi_percent_sig_voxels_pLess0.01.png")
            plt.show()
    display(Markdown("; ".join(match_messages)))
else:
    display(Markdown("_Permutation-cache ROI glass brain was unavailable. No p < .01 Haufe/Z fallback is shown, because saved stage Haufe figures may have been generated with p < .05._"))
    display(Markdown("; ".join(match_messages)))


## 3. Permutation-Significant Voxels (p < .01)
This section reports permutation-importance feature-mask diagnostics and any saved glass-brain/ribbon figures for significant or fallback feature spaces.


In [ ]:
r11imp = results.get("stage11_importance", {})
display_df(flatten_result_dict(r11imp), rows=80)

def _stage11_dict(key_options):
    for key in key_options:
        val = r11imp.get(key) if isinstance(r11imp, dict) else None
        if isinstance(val, dict):
            return val
    return {}

masks = _stage11_dict(["importance_mask_permutated", "importance_masks_permutated", "importance_masks", "mask"])
scores = _stage11_dict(["importance_scores_permutated", "importance_scores", "scores", "actual_importance"])
pvals = _stage11_dict(["p_values_permutated", "p_values", "pvals"])
ACTIVE_IMPORTANCE_MASKS = {group: np.asarray(mask, dtype=bool).copy() for group, mask in masks.items() if mask is not None}
ACTIVE_IMPORTANCE_MASK_LABEL = f"stage-11 p < {IMPORTANCE_P_THRESHOLD:g}"
if ACTIVE_IMPORTANCE_MASKS:
    display(Markdown(f"**Active downstream feature mask:** {ACTIVE_IMPORTANCE_MASK_LABEL}; counts " + str({g: int(m.sum()) for g, m in ACTIVE_IMPORTANCE_MASKS.items()})))
rows = []
for group in ["SAD", "HC"]:
    mask = masks.get(group)
    score = scores.get(group)
    pval = pvals.get(group)
    rows.append({
        "group": group,
        "mask_features_p_lt_01": int(np.asarray(mask, dtype=bool).sum()) if mask is not None else np.nan,
        "p_lt_05_features": int((np.isfinite(np.asarray(pval, dtype=float)) & (np.asarray(pval, dtype=float) < 0.05)).sum()) if pval is not None else np.nan,
        "p05_note": "no voxels survive p < 0.05" if pval is not None and int((np.isfinite(np.asarray(pval, dtype=float)) & (np.asarray(pval, dtype=float) < 0.05)).sum()) == 0 else "",
        "positive_importance": int((np.asarray(score) > 0).sum()) if score is not None else np.nan,
        "max_importance": float(np.nanmax(score)) if score is not None and np.asarray(score).size else np.nan,
        "min_p": float(np.nanmin(pval)) if pval is not None and np.asarray(pval).size else np.nan,
    })
summary = pd.DataFrame(rows)
display_df(summary, rows=10)
if not summary.empty:
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.barplot(data=summary, x="group", y="mask_features_p_lt_01", ax=ax, color="#59A14F")
    ax.set_title(f"Permutation-significant feature counts (p < {IMPORTANCE_P_THRESHOLD:g})")
    ax.set_ylabel(f"Selected voxels, p < {IMPORTANCE_P_THRESHOLD:g}")
    save_current_fig("section03_permutation_feature_counts_pLess0.01.png")
    plt.show()

# Glass brain for all permutation-significant voxels from stage 11.
def _roi_vector_order_stage11():
    # Stage 11 feature vectors follow the ROI concatenation order saved by the L2 script.
    if "_default_roi_order" in globals():
        return list(_default_roi_order())
    if PIPELINE_NAME == "MemoryFearNetwork":
        return ["left_acc", "left_amygdala", "left_hippocampus", "left_insula", "left_vmpfc", "left_VVC", "left_AG", "left_SMG", "left_IFG", "left_MFG", "left_SFG", "left_Precuneus", "right_acc", "right_amygdala", "right_hippocampus", "right_insula", "right_vmpfc", "right_VVC", "right_AG", "right_SMG", "right_IFG", "right_MFG", "right_SFG", "right_Precuneus"]
    return ["left_acc", "left_amygdala", "left_hippocampus", "left_insula", "left_vmpfc", "right_acc", "right_amygdala", "right_hippocampus", "right_insula", "right_vmpfc"]

def _roi_plot_order_stage11():
    return roi_display_order_for_pipeline(_roi_vector_order_stage11())

def _roi_dir_stage11():
    roi_order = _roi_vector_order_stage11()
    if "_find_roi_dir" in globals():
        roi_dir = _find_roi_dir(roi_order)
        if roi_dir is not None:
            return roi_dir
    candidates = _pipeline_roi_candidates() if "_pipeline_roi_candidates" in globals() else [Path(os.environ.get("ROI_DIR", ""))]
    for d in candidates:
        if d and d.exists() and all(list(d.glob(f"*{roi}*.nii*")) for roi in roi_order):
            return d
    return None

def _reconstruct_roi_map(flat_data, roi_order, roi_dir):
    try:
        import nibabel as nib
    except Exception as exc:
        return None, f"nibabel unavailable: {exc}"
    arr = np.asarray(flat_data, dtype=float).ravel()
    ref_img = None
    chunks = []
    for roi in roi_order:
        files = sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))
        if not files:
            return None, f"missing mask for {roi}"
        img = nib.load(str(files[0]))
        if ref_img is None:
            ref_img = img
        chunks.append(int((img.get_fdata() > 0).sum()))
    if sum(chunks) != arr.size:
        return None, f"ROI voxel count {sum(chunks)} does not match vector length {arr.size}"
    vol = np.zeros(ref_img.shape, dtype=float)
    cursor = 0
    for roi, n in zip(roi_order, chunks):
        img = nib.load(str(sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))[0]))
        mask_data = img.get_fdata() > 0
        vol[mask_data] = arr[cursor: cursor + n]
        cursor += n
    return nib.Nifti1Image(vol, ref_img.affine, ref_img.header), None

def _roi_sig_print(mask, roi_order, roi_dir, group):
    import nibabel as nib
    arr = np.asarray(mask, dtype=bool).ravel()
    cursor = 0
    rows = []
    for roi in roi_order:
        img = nib.load(str(sorted(Path(roi_dir).glob(f"*{roi}*.nii*"))[0]))
        n = int((img.get_fdata() > 0).sum())
        selected = int(arr[cursor: cursor + n].sum())
        if selected > 0:
            rows.append({"Group": group, "ROI": roi, "ROI_voxels": n, "Selected_voxels": selected, "Percent_ROI": 100 * selected / n})
        cursor += n
    print(f"\nPermutation-significant voxels for {group} (p < {IMPORTANCE_P_THRESHOLD:g}):")
    print(f"{'ROI Name':<25} | {'Voxels':<8} | {'% ROI'}")
    print("-" * 50)
    row_df = pd.DataFrame(rows)
    if not row_df.empty:
        row_order = roi_display_order_for_pipeline(row_df["ROI"].tolist())
        row_df["ROI_order"] = pd.Categorical(row_df["ROI"], categories=row_order, ordered=True)
        row_df = row_df.sort_values("ROI_order")
        for _, row in row_df.iterrows():
            print(f"{row['ROI']:<25} | {int(row['Selected_voxels']):<8} | {row['Percent_ROI']:.2f}%")
        return row_df.drop(columns=["ROI_order"])
    return row_df

roi_order = _roi_vector_order_stage11()
roi_plot_order = _roi_plot_order_stage11()
roi_dir = _roi_dir_stage11()
if roi_dir is None:
    display(Markdown("_Could not locate MemoryFearNetwork ROI masks for stage 11 glass brain._"))
else:
    try:
        from nilearn import plotting
        fig = plt.figure(figsize=(10, 8))
        axes = {"SAD": fig.add_subplot(2, 1, 1), "HC": fig.add_subplot(2, 1, 2)}
        roi_tables = []
        for group, ax in axes.items():
            mask = np.asarray(masks.get(group), dtype=bool) if group in masks else None
            score = np.asarray(scores.get(group), dtype=float) if group in scores else None
            if mask is None or score is None:
                ax.axis('off')
                ax.text(0.5, 0.5, f"{group}: missing mask or score", ha='center', va='center')
                continue
            values = np.where(mask, score, 0.0)
            img, err = _reconstruct_roi_map(values, roi_order, roi_dir)
            if img is None:
                ax.axis('off')
                ax.text(0.5, 0.5, f"{group}: {err}", ha='center', va='center', wrap=True)
                continue
            nonzero = np.abs(values[np.isfinite(values) & (values != 0)])
            threshold = float(np.min(nonzero)) if nonzero.size else 0.0
            vmax = float(np.percentile(nonzero, 99)) if nonzero.size else 1.0
            plotting.plot_glass_brain(img, display_mode='lyrz', colorbar=True, threshold=threshold, vmax=vmax, cmap='RdBu_r', plot_abs=False, black_bg=False, axes=ax, title=f"{group}: stage 11 permutation-significant voxels (p < {IMPORTANCE_P_THRESHOLD:g})")
            roi_tables.append(_roi_sig_print(mask, roi_order, roi_dir, group))
        save_current_fig("section03_permutation_sig_voxels_glass_brain_pLess0.01.png")
        plt.show()
        if roi_tables:
            roi_df = pd.concat(roi_tables, ignore_index=True)
            roi_order_plot_present = [roi for roi in roi_plot_order if roi in set(roi_df["ROI"])]
            roi_df["ROI_label"] = roi_df["ROI"].map(roi_display_label)
            roi_order_labels = [roi_display_label(roi) for roi in roi_order_plot_present]
            fig_roi, ax_roi = plt.subplots(figsize=(9, max(6, 0.42 * len(roi_order_labels))))
            sns.barplot(data=roi_df, y="ROI_label", x="Percent_ROI", hue="Group", order=roi_order_labels, palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=ax_roi)
            for container in ax_roi.containers:
                labels = [f"{bar.get_width():.1f}%" if np.isfinite(bar.get_width()) and bar.get_width() > 0 else "" for bar in container]
                ax_roi.bar_label(container, labels=labels, padding=3, fontsize=8)
            ax_roi.set_xlim(0, max(roi_df["Percent_ROI"].max() * 1.35, 1))
            ax_roi.set_xlabel("Significant voxels (% of ROI)")
            ax_roi.set_ylabel("")
            ax_roi.tick_params(axis='y', labelsize=9)
            ax_roi.set_title(f"ROI distribution of stage 11 permutation-significant voxels (p < {IMPORTANCE_P_THRESHOLD:g})")
            save_current_fig("section03_permutation_sig_voxels_roi_percent_pLess0.01.png")
            plt.show()
    except Exception as exc:
        display(Markdown(f"_Could not render stage 11 glass brain: {exc}_"))


Important note: section 3 reconstructs the stricter p < .01 stage-11 mask from saved p-values. Sections 4+ below now recompute the main downstream visualization tables in memory with that active mask, while leaving the saved `.joblib` files unchanged.


## Active p < .01 Feature-Space Recompute
From this point onward, the notebook rebuilds the section-4+ analysis tables in memory using `ACTIVE_IMPORTANCE_MASKS` from section 3. Saved `.joblib` payloads are still used as raw inputs/checkpoints, but downstream RDMs, drift, trajectories, and decision-boundary summaries below are regenerated with the p < .01 feature masks.


In [ ]:
def _active_importance_counts():
    return {g: int(np.asarray(m, dtype=bool).sum()) for g, m in globals().get("ACTIVE_IMPORTANCE_MASKS", {}).items()}


def _collect_feature_space_dicts(obj):
    found = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            key_l = str(key).lower()
            if "feature_space" in key_l and isinstance(value, dict):
                found.append(value)
            found.extend(_collect_feature_space_dicts(value))
    elif isinstance(obj, (list, tuple)):
        for value in obj:
            found.extend(_collect_feature_space_dicts(value))
    return found


def _feature_space_counts(feature_space):
    counts = {}
    if not isinstance(feature_space, dict):
        return counts
    for group in ["SAD", "HC"]:
        entry = feature_space.get(group) or feature_space.get(group.lower())
        if isinstance(entry, dict):
            for key in ["selected_features", "n_features", "features", "mask_features"]:
                if key in entry and pd.notna(entry[key]):
                    try:
                        counts[group] = int(entry[key])
                        break
                    except Exception:
                        pass
        elif entry is not None:
            try:
                counts[group] = int(entry)
            except Exception:
                pass
    return counts


def _payload_matches_active_importance(payload, label):
    active = _active_importance_counts()
    if not active:
        display(Markdown(f"_Skipping {label}: ACTIVE_IMPORTANCE_MASKS is not defined. Run section 3 first._"))
        return False
    feature_spaces = _collect_feature_space_dicts(payload)
    if not feature_spaces:
        display(Markdown(f"_Skipping {label}: saved payload has no feature-space metadata, so the notebook cannot verify it used {globals().get('ACTIVE_IMPORTANCE_MASK_LABEL', 'the active mask')}._"))
        return False
    for feature_space in feature_spaces:
        counts = _feature_space_counts(feature_space)
        if counts and all(counts.get(group) == active.get(group) for group in active):
            display(Markdown(f"**{label} uses active feature mask:** {globals().get('ACTIVE_IMPORTANCE_MASK_LABEL', 'active mask')} counts {active}."))
            return True
    summarized = [_feature_space_counts(fs) for fs in feature_spaces]
    display(Markdown(f"_Skipping {label}: saved feature-space counts {summarized} do not match active mask counts {active}._"))
    return False




# Notebook-side active-mask recomputation for sections 4+.
# This does not modify the analysis .py files or saved joblib checkpoints.
from sklearn.covariance import LedoitWolf
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ACTIVE_RECOMPUTED_RESULTS = {}


def _feature_space_from_active():
    masks = globals().get("ACTIVE_IMPORTANCE_MASKS", {})
    return {
        group: {
            "source": globals().get("ACTIVE_IMPORTANCE_MASK_LABEL", "ACTIVE_IMPORTANCE_MASKS"),
            "selected_features": int(np.asarray(mask, dtype=bool).sum()),
            "mask_rule": globals().get("ACTIVE_IMPORTANCE_MASK_LABEL", "ACTIVE_IMPORTANCE_MASKS"),
        }
        for group, mask in masks.items()
    }


def _load_active_base_inputs():
    r03_local = results.get("stage03_data", {})
    r05_local = load_first([CHECKPOINT_DIR / "cell_05.joblib", LEGACY_CHECKPOINT_DIR / "cell_05.joblib"], default={})
    r05_local = coerce_result(r05_local)
    if not isinstance(r03_local, dict) or not isinstance(r05_local, dict):
        return None, "cell_03/cell_05 payloads were not available"
    required_03 = {"X_ext", "y_ext", "sub_ext", "X_reinst", "y_reinst", "sub_reinst"}
    if not required_03.issubset(r03_local.keys()):
        return None, f"cell_03 is missing {sorted(required_03.difference(r03_local.keys()))}"
    if "data_subsets" not in r05_local:
        return None, "cell_05 has no data_subsets"
    return {"r03": r03_local, "r05": r05_local, "data_subsets": r05_local["data_subsets"], "meta": r05_local.get("meta"), "sub_to_meta": r05_local.get("sub_to_meta", {})}, ""


def _as_mask(group, n_features):
    masks = globals().get("ACTIVE_IMPORTANCE_MASKS", {})
    if group not in masks:
        raise KeyError(f"ACTIVE_IMPORTANCE_MASKS has no {group} mask")
    mask = np.asarray(masks[group], dtype=bool).ravel()
    if mask.size != n_features:
        raise ValueError(f"{group} active mask has {mask.size} features but data have {n_features}")
    if int(mask.sum()) < 1:
        raise ValueError(f"{group} active mask has zero selected features")
    return mask


def _phase_data(data_subsets, group, drug, phase):
    base = globals().get("_ACTIVE_BASE_INPUTS_FOR_PHASE")
    phase_key = "ext" if phase.lower().startswith("ext") else "rst"
    if isinstance(base, dict) and "r03" in base:
        r03 = base["r03"]
        X = np.asarray(r03["X_ext"] if phase_key == "ext" else r03["X_reinst"])
        y = np.asarray(r03["y_ext"] if phase_key == "ext" else r03["y_reinst"])
        sub = np.asarray(r03["sub_ext"] if phase_key == "ext" else r03["sub_reinst"])
        sub_to_meta = base.get("sub_to_meta", {}) or {}
        keep = []
        for s in sub:
            info = sub_to_meta.get(str(s), {})
            keep.append(str(info.get("Group", "")).upper() == str(group).upper() and str(info.get("Drug", "")).lower() == str(drug).lower())
        keep = np.asarray(keep, dtype=bool)
        return {"X": X[keep], "y": y[keep], "sub": sub[keep]}
    key = f"{group}_{drug}"
    return data_subsets[key][phase_key]

SHOCK_TARGET_LABELS_NOTEBOOK = ["Shock", "SHOCK", "shock", "US", "UCS", "unconditioned_stimulus", "Shock1", "Shock2", "Shock3", "US1", "US2", "US3"]


def _condition_mask_notebook(labels, condition):
    if isinstance(condition, (list, tuple, set, np.ndarray)):
        return np.isin(labels, list(condition))
    return labels == condition


def _shock_target_data(base, group, drug):
    if not isinstance(base, dict) or "r03" not in base:
        return None
    r03 = base["r03"]
    sub_to_meta = base.get("sub_to_meta", {}) or {}
    parts = []
    for x_key, y_key, s_key in [("X_ext_all", "y_ext_all", "sub_ext_all"), ("X_reinst_all", "y_reinst_all", "sub_reinst_all"), ("X_ext", "y_ext", "sub_ext"), ("X_reinst", "y_reinst", "sub_reinst")]:
        if not all(k in r03 for k in [x_key, y_key, s_key]):
            continue
        X = np.asarray(r03[x_key])
        y = np.asarray(r03[y_key])
        sub = np.asarray(r03[s_key])
        keep_group = []
        for s in sub:
            info = sub_to_meta.get(str(s), {}) or sub_to_meta.get(f"sub-{s}", {}) or sub_to_meta.get(str(s).replace("sub-", ""), {})
            keep_group.append(str(info.get("Group", "")).upper() == str(group).upper() and str(info.get("Drug", "")).lower() == str(drug).lower())
        keep = _condition_mask_notebook(y, SHOCK_TARGET_LABELS_NOTEBOOK) & np.asarray(keep_group, dtype=bool)
        if np.any(keep):
            parts.append({"X": X[keep], "y": y[keep], "sub": sub[keep]})
    if not parts:
        return None
    return {"X": np.vstack([p["X"] for p in parts]), "y": np.concatenate([p["y"] for p in parts]), "sub": np.concatenate([p["sub"] for p in parts])}


def _subject_ids(data):
    return np.array(sorted(pd.Series(data["sub"]).dropna().unique()))


def _mean_for(data, condition, mask, sub=None):
    y = np.asarray(data["y"])
    X = np.asarray(data["X"])
    selector = y == condition
    if sub is not None:
        selector = selector & (np.asarray(data["sub"]) == sub)
    if not np.any(selector):
        return None
    return np.nanmean(X[selector][:, mask], axis=0)


def _condition_trial_rows(data, condition, mask, group, domain, source_mean, target_mean):
    rows = []
    vec = target_mean - source_mean
    denom = float(np.dot(vec, vec)) + 1e-12
    for sub in _subject_ids(data):
        idx = np.where((np.asarray(data["sub"]) == sub) & (np.asarray(data["y"]) == condition))[0]
        for trial_num, row_idx in enumerate(idx, start=1):
            x = np.asarray(data["X"])[row_idx, :][:, None].ravel()[mask]
            score = float(np.dot(x - source_mean, vec) / denom)
            rows.append({"sub": sub, "Group": group, "Condition": domain, "trial": trial_num, "score": score})
    return rows


def _cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan


def _projection(delta, ideal):
    denom = np.linalg.norm(ideal)
    return float(np.dot(delta, ideal) / denom) if denom > 0 else np.nan



def _residualized_shock_anchor_projection_notebook(cs_data, shock_data, mask):
    rows = []
    if not isinstance(cs_data, dict) or not isinstance(shock_data, dict):
        return pd.DataFrame(rows)
    X_cs = np.asarray(cs_data.get("X"))[:, mask]
    y_cs = np.asarray(cs_data.get("y"))
    sub_cs = np.asarray(cs_data.get("sub"))
    X_shock = np.asarray(shock_data.get("X"))[:, mask]
    sub_shock = np.asarray(shock_data.get("sub"))
    if X_cs.shape[1] == 0 or X_shock.shape[0] == 0:
        return pd.DataFrame(rows)
    X_cs = X_cs - np.mean(X_cs, axis=1, keepdims=True)
    X_shock = X_shock - np.mean(X_shock, axis=1, keepdims=True)
    for sub in np.intersect1d(np.unique(sub_cs), np.unique(sub_shock)):
        m_base = (sub_cs == sub) & _condition_mask_notebook(y_cs, "CS-")
        m_css = (sub_cs == sub) & _condition_mask_notebook(y_cs, "CSS")
        m_csr = (sub_cs == sub) & _condition_mask_notebook(y_cs, "CSR")
        m_shock = (sub_shock == sub)
        if not (np.any(m_base) and np.any(m_css) and np.any(m_csr) and np.any(m_shock)):
            continue
        base = np.mean(X_cs[m_base], axis=0)
        shock = np.mean(X_shock[m_shock], axis=0)
        axis = shock - base
        axis_norm = np.linalg.norm(axis)
        if not np.isfinite(axis_norm) or axis_norm <= 0:
            continue
        axis = axis / axis_norm
        def _proj(vec):
            delta = vec - base
            delta_norm = np.linalg.norm(delta)
            projection = float(np.dot(delta, axis))
            cosine = np.nan if delta_norm <= 0 else float(projection / delta_norm)
            return projection, cosine
        proj_css, cos_css = _proj(np.mean(X_cs[m_css], axis=0))
        proj_csr, cos_csr = _proj(np.mean(X_cs[m_csr], axis=0))
        rows.append({"Subject": sub, "Projection_CSS": proj_css, "Projection_CSR": proj_csr, "Projection_CSR_minus_CSS": proj_csr - proj_css, "Cosine_CSS": cos_css, "Cosine_CSR": cos_csr, "Cosine_CSR_minus_CSS": cos_csr - cos_css if np.isfinite(cos_css) and np.isfinite(cos_csr) else np.nan, "Shock_Axis_Norm": float(axis_norm), "n_shock_trials": int(np.sum(m_shock))})
    return pd.DataFrame(rows)


def _summarize_shock_anchor_notebook(df_sad, df_hc):
    metrics = ["Projection_CSS", "Projection_CSR", "Projection_CSR_minus_CSS", "Cosine_CSS", "Cosine_CSR", "Cosine_CSR_minus_CSS", "Shock_Axis_Norm"]
    out = {}
    for metric in metrics:
        if metric not in df_sad.columns or metric not in df_hc.columns:
            continue
        sad_vals = pd.to_numeric(df_sad[metric], errors="coerce").dropna().to_numpy()
        hc_vals = pd.to_numeric(df_hc[metric], errors="coerce").dropna().to_numpy()
        if len(sad_vals) < 2 or len(hc_vals) < 2:
            out[metric] = {"SAD": sad_vals, "HC": hc_vals, "means": {"SAD": np.nan, "HC": np.nan}, "group_stats": (np.nan, np.nan)}
            continue
        t_val, p_val = stats.ttest_ind(sad_vals, hc_vals, equal_var=False, nan_policy="omit")[:2]
        out[metric] = {"SAD": sad_vals, "HC": hc_vals, "means": {"SAD": float(np.nanmean(sad_vals)), "HC": float(np.nanmean(hc_vals))}, "group_stats": (float(t_val), float(p_val))}
    return out


def _crossnobis_subject_rdms(data, mask, conditions, zscore=False):
    X = np.asarray(data["X"], dtype=float)[:, mask]
    y = np.asarray(data["y"])
    subs = np.asarray(data["sub"])
    if zscore:
        mu = np.nanmean(X, axis=0)
        sd = np.nanstd(X, axis=0)
        sd[sd == 0] = 1.0
        X = (X - mu) / sd
    rdms = []
    used_subs = []
    for sub in sorted(pd.Series(subs).dropna().unique()):
        sub_idx = np.where(subs == sub)[0]
        means_a = {}
        means_b = {}
        residuals = []
        ok = True
        for cond in conditions:
            idx = sub_idx[y[sub_idx] == cond]
            if len(idx) < 2:
                ok = False
                break
            half = max(1, len(idx) // 2)
            idx_a = idx[:half]
            idx_b = idx[half:]
            if len(idx_b) == 0:
                idx_b = idx_a
            means_a[cond] = np.nanmean(X[idx_a], axis=0)
            means_b[cond] = np.nanmean(X[idx_b], axis=0)
            cond_mean = np.nanmean(X[idx], axis=0)
            residuals.append(X[idx] - cond_mean)
        if not ok:
            continue
        resid = np.vstack(residuals)
        if resid.shape[0] <= 2 or resid.shape[1] < 1:
            continue
        try:
            cov = LedoitWolf().fit(resid).covariance_
            inv_cov = np.linalg.pinv(cov)
        except Exception:
            inv_cov = np.eye(resid.shape[1])
        mat = np.zeros((len(conditions), len(conditions)), dtype=float)
        for i, ca in enumerate(conditions):
            for j, cb in enumerate(conditions):
                if i == j:
                    continue
                d1 = means_a[ca] - means_a[cb]
                d2 = means_b[ca] - means_b[cb]
                mat[i, j] = float(d1 @ inv_cov @ d2 / max(1, resid.shape[1]))
        mat = (mat + mat.T) / 2.0
        rdms.append(mat)
        used_subs.append(sub)
    return np.asarray(rdms), np.asarray(used_subs)


def _topology_stats(rdms_sad, rdms_hc, i_csr=2, i_css=1, i_csminus=0):
    vA_sad = rdms_sad[:, i_csr, i_css]
    vA_hc = rdms_hc[:, i_csr, i_css]
    vB_sad = rdms_sad[:, i_css, i_csminus]
    vB_hc = rdms_hc[:, i_css, i_csminus]
    return {
        "metric_a_stats_raw": stats.ttest_ind(vA_sad, vA_hc, equal_var=False, nan_policy="omit"),
        "metric_b_stats_raw": stats.ttest_ind(vB_sad, vB_hc, equal_var=False, nan_policy="omit"),
        "one_sample_stats_raw": {
            "p_a_sad": stats.ttest_1samp(vA_sad, 0, nan_policy="omit").pvalue,
            "p_a_hc": stats.ttest_1samp(vA_hc, 0, nan_policy="omit").pvalue,
            "p_b_sad": stats.ttest_1samp(vB_sad, 0, nan_policy="omit").pvalue,
            "p_b_hc": stats.ttest_1samp(vB_hc, 0, nan_policy="omit").pvalue,
        },
    }


def _recompute_stage12(base):
    data_subsets = base["data_subsets"]
    sad = _phase_data(data_subsets, "SAD", "Placebo", "ext")
    hc = _phase_data(data_subsets, "HC", "Placebo", "ext")
    n_features = np.asarray(sad["X"]).shape[1]
    mask_sad = _as_mask("SAD", n_features)
    mask_hc = _as_mask("HC", n_features)
    conditions = ["CS-", "CSS", "CSR"]
    rdms_sad_raw, subs_sad = _crossnobis_subject_rdms(sad, mask_sad, conditions, zscore=False)
    rdms_hc_raw, subs_hc = _crossnobis_subject_rdms(hc, mask_hc, conditions, zscore=False)
    rdms_sad_z, _ = _crossnobis_subject_rdms(sad, mask_sad, conditions, zscore=True)
    rdms_hc_z, _ = _crossnobis_subject_rdms(hc, mask_hc, conditions, zscore=True)
    n_sad = max(1, int(mask_sad.sum()))
    n_hc = max(1, int(mask_hc.sum()))
    shock_sad = _shock_target_data(globals().get("_ACTIVE_BASE_INPUTS_FOR_PHASE"), "SAD", "Placebo")
    shock_hc = _shock_target_data(globals().get("_ACTIVE_BASE_INPUTS_FOR_PHASE"), "HC", "Placebo")
    shock_anchor_df_sad = pd.DataFrame()
    shock_anchor_df_hc = pd.DataFrame()
    shock_anchor_df = pd.DataFrame()
    shock_anchor_stats = {}
    if isinstance(shock_sad, dict) and isinstance(shock_hc, dict) and len(shock_sad.get("X", [])) and len(shock_hc.get("X", [])):
        shock_anchor_df_sad = _residualized_shock_anchor_projection_notebook(sad, shock_sad, mask_sad)
        shock_anchor_df_hc = _residualized_shock_anchor_projection_notebook(hc, shock_hc, mask_hc)
        if not shock_anchor_df_sad.empty:
            shock_anchor_df_sad["Group"] = "SAD"
        if not shock_anchor_df_hc.empty:
            shock_anchor_df_hc["Group"] = "HC"
        shock_anchor_df = pd.concat([shock_anchor_df_sad, shock_anchor_df_hc], ignore_index=True)
        shock_anchor_stats = _summarize_shock_anchor_notebook(shock_anchor_df_sad, shock_anchor_df_hc)
    results_12 = {
        "RDM_CONDITIONS": conditions,
        "rdms_sad_raw": rdms_sad_raw,
        "rdms_hc_raw": rdms_hc_raw,
        "rdms_sad_z": rdms_sad_z,
        "rdms_hc_z": rdms_hc_z,
        "rdms_sad_raw_pv": rdms_sad_raw / n_sad,
        "rdms_hc_raw_pv": rdms_hc_raw / n_hc,
        "rdms_sad_z_pv": rdms_sad_z / n_sad,
        "rdms_hc_z_pv": rdms_hc_z / n_hc,
        "subs_sad_rdm": subs_sad,
        "subs_hc_rdm": subs_hc,
        "shock_target_labels": SHOCK_TARGET_LABELS_NOTEBOOK,
        "shock_anchor_description": "Trial-wise feature-mean residualized projection of CSS/CSR onto each subject's Shock-minus-CS- axis; Shock is not included in the primary RDM.",
        "shock_anchor_df": shock_anchor_df,
        "shock_anchor_stats": shock_anchor_stats,
        "shock_anchor_results": {"df": shock_anchor_df, "SAD": shock_anchor_df_sad, "HC": shock_anchor_df_hc, "stats": shock_anchor_stats},
        "mask_sad_analysis": mask_sad,
        "mask_hc_analysis": mask_hc,
        "feature_space": _feature_space_from_active(),
        "feature_space_12": _feature_space_from_active(),
        "active_recomputed": True,
    }
    results_12.update(_topology_stats(rdms_sad_raw, rdms_hc_raw))
    z_stats = _topology_stats(rdms_sad_z, rdms_hc_z)
    results_12["metric_a_stats_z"] = z_stats["metric_a_stats_raw"]
    results_12["metric_b_stats_z"] = z_stats["metric_b_stats_raw"]
    results_12["one_sample_stats_z"] = {k.replace("raw", "z"): v for k, v in z_stats["one_sample_stats_raw"].items()}
    pv_stats = _topology_stats(results_12["rdms_sad_raw_pv"], results_12["rdms_hc_raw_pv"])
    results_12["metric_a_stats_raw_pv"] = pv_stats["metric_a_stats_raw"]
    results_12["metric_b_stats_raw_pv"] = pv_stats["metric_b_stats_raw"]
    results_12["one_sample_stats_raw_pv"] = pv_stats["one_sample_stats_raw"]
    return {"results_12": results_12, "feature_space": results_12["feature_space"], "active_recomputed": True}


def _drift_rows_for_group(data_subsets, group, drug, mask):
    ext = _phase_data(data_subsets, group, drug, "ext")
    rst = _phase_data(data_subsets, group, drug, "rst")
    shock = _shock_target_data(globals().get("_ACTIVE_BASE_INPUTS_FOR_PHASE"), group, drug)
    safety_source = _mean_for(ext, "CSS", mask)
    safety_target = _mean_for(ext, "CS-", mask)
    threat_source = _mean_for(ext, "CSR", mask)
    threat_target = _mean_for(rst, "CSR", mask)
    shock_target = np.nanmean(np.asarray(shock["X"])[:, mask], axis=0) if isinstance(shock, dict) and len(shock.get("X", [])) else None
    ideal_safety = safety_target - safety_source if safety_source is not None and safety_target is not None else None
    ideal_threat = threat_target - threat_source if threat_source is not None and threat_target is not None else None
    ideal_shock = shock_target - threat_source if threat_source is not None and shock_target is not None else None
    rows = []
    common_subs = sorted(set(_subject_ids(ext)).intersection(set(_subject_ids(rst))))
    for sub in common_subs:
        s_css = _mean_for(ext, "CSS", mask, sub=sub)
        s_csm = _mean_for(ext, "CS-", mask, sub=sub)
        s_csr_ext = _mean_for(ext, "CSR", mask, sub=sub)
        s_csr_rst = _mean_for(rst, "CSR", mask, sub=sub)
        if s_css is not None and s_csm is not None and ideal_safety is not None:
            delta = s_csm - s_css
            rows.append({"sub": sub, "Group": group, "Drug": drug, "Condition": "Safety", "Domain": "Safety", "projection": _projection(delta, ideal_safety), "cosine": _cosine(delta, ideal_safety), "init_dist": float(np.linalg.norm(s_css - safety_target))})
        if s_csr_ext is not None and s_csr_rst is not None and ideal_threat is not None:
            delta = s_csr_rst - s_csr_ext
            rows.append({"sub": sub, "Group": group, "Drug": drug, "Condition": "Threat", "Domain": "Threat", "projection": _projection(delta, ideal_threat), "cosine": _cosine(delta, ideal_threat), "init_dist": float(np.linalg.norm(s_csr_ext - threat_target))})
        if s_csr_ext is not None and ideal_shock is not None and isinstance(shock, dict):
            shock_sub = _mean_for(shock, SHOCK_TARGET_LABELS_NOTEBOOK, mask, sub=sub)
            if shock_sub is not None:
                delta = shock_sub - s_csr_ext
                rows.append({"sub": sub, "Group": group, "Drug": drug, "Condition": "Threat Shock Target", "Domain": "Threat Shock Target", "projection": _projection(delta, ideal_shock), "cosine": _cosine(delta, ideal_shock), "init_dist": float(np.linalg.norm(s_csr_ext - shock_target))})
    return rows


def _recompute_stage13(base):
    data_subsets = base["data_subsets"]
    n_features = np.asarray(_phase_data(data_subsets, "SAD", "Placebo", "ext")["X"]).shape[1]
    rows = []
    rows.extend(_drift_rows_for_group(data_subsets, "SAD", "Placebo", _as_mask("SAD", n_features)))
    rows.extend(_drift_rows_for_group(data_subsets, "HC", "Placebo", _as_mask("HC", n_features)))
    df_plot = pd.DataFrame(rows)
    summary_rows = []
    for metric in ["projection", "cosine"]:
        for cond in ["Safety", "Threat", "Threat Shock Target"]:
            sad = pd.to_numeric(df_plot.loc[(df_plot["Group"] == "SAD") & (df_plot["Condition"] == cond), metric], errors="coerce").dropna()
            hc = pd.to_numeric(df_plot.loc[(df_plot["Group"] == "HC") & (df_plot["Condition"] == cond), metric], errors="coerce").dropna()
            t, p = stats.ttest_ind(sad, hc, equal_var=False, nan_policy="omit") if len(sad) > 1 and len(hc) > 1 else (np.nan, np.nan)
            summary_rows.append({"metric": metric, "Condition": cond, "SAD_mean": sad.mean(), "HC_mean": hc.mean(), "t": t, "p": p})
    results_13 = {"df_plot": df_plot, "drift_summary": pd.DataFrame(summary_rows), "primary_metric": "projection", "feature_space": _feature_space_from_active(), "active_recomputed": True}
    return {"results_13": results_13, "feature_space": results_13["feature_space"], "active_recomputed": True}


def _trajectory_stats(df):
    rows = []
    if df.empty:
        return pd.DataFrame()
    for trial, sub in df.groupby("trial"):
        sad = pd.to_numeric(sub.loc[sub["Group"] == "SAD", "score"], errors="coerce").dropna()
        hc = pd.to_numeric(sub.loc[sub["Group"] == "HC", "score"], errors="coerce").dropna()
        t, p = stats.ttest_ind(sad, hc, equal_var=False, nan_policy="omit") if len(sad) > 1 and len(hc) > 1 else (np.nan, np.nan)
        rows.append({"Trial": trial, "SAD_Mean": sad.mean(), "HC_Mean": hc.mean(), "Diff_t": t, "Diff_p": p, "n_SAD": len(sad), "n_HC": len(hc)})
    out = pd.DataFrame(rows).sort_values("Trial")
    if not out.empty:
        out["Diff_p_fdr"] = bh_fdr(out["Diff_p"].to_numpy())
    return out


def _recompute_stage14(base):
    data_subsets = base["data_subsets"]
    n_features = np.asarray(_phase_data(data_subsets, "SAD", "Placebo", "ext")["X"]).shape[1]
    safe_rows = []
    threat_rows = []
    threat_shock_rows = []
    for group in ["SAD", "HC"]:
        mask = _as_mask(group, n_features)
        ext = _phase_data(data_subsets, group, "Placebo", "ext")
        rst = _phase_data(data_subsets, group, "Placebo", "rst")
        shock = _shock_target_data(base, group, "Placebo")
        safe_source = _mean_for(ext, "CSS", mask)
        safe_target = _mean_for(ext, "CS-", mask)
        threat_source = _mean_for(ext, "CSR", mask)
        threat_target = _mean_for(rst, "CSR", mask)
        shock_target = np.nanmean(np.asarray(shock["X"])[:, mask], axis=0) if isinstance(shock, dict) and len(shock.get("X", [])) else None
        safe_rows.extend(_condition_trial_rows(ext, "CSS", mask, group, "Safety", safe_source, safe_target))
        threat_rows.extend(_condition_trial_rows(ext, "CSR", mask, group, "Threat", threat_source, threat_target))
        if shock_target is not None:
            threat_shock_rows.extend(_condition_trial_rows(ext, "CSR", mask, group, "Threat Shock Target", threat_source, shock_target))
    df_safe = pd.DataFrame(safe_rows)
    df_threat = pd.DataFrame(threat_rows)
    df_threat_shock = pd.DataFrame(threat_shock_rows)
    slope_rows = []
    for name, df in [("Safety", df_safe), ("Threat", df_threat), ("Threat Shock Target", df_threat_shock)]:
        if df.empty:
            continue
        for (group, sub), sdf in df.groupby(["Group", "sub"]):
            if sdf["trial"].nunique() > 1:
                slope = np.polyfit(pd.to_numeric(sdf["trial"], errors="coerce"), pd.to_numeric(sdf["score"], errors="coerce"), 1)[0]
            else:
                slope = np.nan
            slope_rows.append({"Group": group, "sub": sub, "Condition": name, "slope": slope})
    results_13_2 = {"stats_safe": _trajectory_stats(df_safe), "stats_threat": _trajectory_stats(df_threat), "stats_threat_shock": _trajectory_stats(df_threat_shock), "data_safe": df_safe, "data_threat": df_threat, "data_threat_shock": df_threat_shock, "trajectory_slopes": pd.DataFrame(slope_rows), "primary_metric": "score", "feature_space": _feature_space_from_active(), "shock_target_labels": SHOCK_TARGET_LABELS_NOTEBOOK, "active_recomputed": True}
    return {"results_13_2": results_13_2, "feature_space": results_13_2["feature_space"], "active_recomputed": True}


def _entropy_binary(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1 - 1e-6)
    return -(p * np.log2(p) + (1 - p) * np.log2(1 - p))


def _decision_rows_for_group(data_subsets, group, mask, include_all_drugs=False):
    train = _phase_data(data_subsets, group, "Placebo", "ext")
    train_sel = np.isin(np.asarray(train["y"]), ["CSS", "CSR"])
    X_train_all = np.asarray(train["X"])[train_sel][:, mask]
    y_train_all = (np.asarray(train["y"])[train_sel] == "CSR").astype(int)
    sub_train_all = np.asarray(train["sub"])[train_sel]
    rows = []
    drugs = ["Placebo", "Oxytocin"] if include_all_drugs else ["Placebo"]
    for drug in drugs:
        data = _phase_data(data_subsets, group, drug, "ext")
        for sub in _subject_ids(data):
            test_sel = (np.asarray(data["sub"]) == sub) & np.isin(np.asarray(data["y"]), ["CSS", "CSR"])
            if not np.any(test_sel):
                continue
            if drug == "Placebo":
                train_keep = sub_train_all != sub
            else:
                train_keep = np.ones(len(sub_train_all), dtype=bool)
            if len(np.unique(y_train_all[train_keep])) < 2:
                continue
            model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, solver="liblinear", C=1.0, class_weight="balanced"))
            model.fit(X_train_all[train_keep], y_train_all[train_keep])
            X_test = np.asarray(data["X"])[test_sel][:, mask]
            y_test_labels = np.asarray(data["y"])[test_sel]
            p_csr = model.predict_proba(X_test)[:, 1]
            p_css = p_csr[y_test_labels == "CSS"]
            p_csr_trials = p_csr[y_test_labels == "CSR"]
            margin_css = np.abs(p_css - 0.5) if p_css.size else np.array([np.nan])
            margin_all = np.abs(p_csr - 0.5) if p_csr.size else np.array([np.nan])
            boundary_sep = np.nanmean(p_csr_trials) - np.nanmean(p_css) if p_css.size and p_csr_trials.size else np.nan
            rows.append({"sub": sub, "Subject": sub, "Group": group, "Drug": drug, "entropy": float(np.nanmean(_entropy_binary(p_csr))), "kurtosis": float(stats.kurtosis(p_csr, nan_policy="omit")) if len(p_csr) > 3 else np.nan, "variance": float(np.nanvar(p_csr)), "probabilities": p_csr, "p_csr_css": float(np.nanmean(p_css)) if p_css.size else np.nan, "p_csr_csr": float(np.nanmean(p_csr_trials)) if p_csr_trials.size else np.nan, "boundary_separation": float(boundary_sep), "decision_margin_css": float(np.nanmean(margin_css)), "decision_margin_all": float(np.nanmean(margin_all))})
    return pd.DataFrame(rows)


def _recompute_stage15_and_19(base):
    data_subsets = base["data_subsets"]
    n_features = np.asarray(_phase_data(data_subsets, "SAD", "Placebo", "ext")["X"]).shape[1]
    df_sad = _decision_rows_for_group(data_subsets, "SAD", _as_mask("SAD", n_features), include_all_drugs=False)
    df_hc = _decision_rows_for_group(data_subsets, "HC", _as_mask("HC", n_features), include_all_drugs=False)
    results_14_self = {"df_sad": df_sad, "df_hc": df_hc, "feature_space": _feature_space_from_active(), "active_recomputed": True}
    df_all = pd.concat([
        _decision_rows_for_group(data_subsets, "SAD", _as_mask("SAD", n_features), include_all_drugs=True),
        _decision_rows_for_group(data_subsets, "HC", _as_mask("HC", n_features), include_all_drugs=True),
    ], ignore_index=True)
    rename = {"entropy": "Entropy", "kurtosis": "Kurtosis", "variance": "Variance", "p_csr_css": "P_CSR_CSS", "p_csr_csr": "P_CSR_CSR", "boundary_separation": "Boundary_Separation", "decision_margin_css": "Decision_Margin_CSS", "decision_margin_all": "Decision_Margin_All"}
    df_open = df_all.rename(columns=rename)
    keep = ["Subject", "Group", "Drug", "Entropy", "Kurtosis", "Variance", "P_CSR_CSS", "P_CSR_CSR", "Boundary_Separation", "Decision_Margin_CSS", "Decision_Margin_All"]
    df_open = df_open[[c for c in keep if c in df_open.columns]].copy()
    stats_open = {metric: _interaction_p_simple(df_open, metric) for metric in ["Entropy", "Kurtosis", "Variance", "P_CSR_CSS", "P_CSR_CSR", "Boundary_Separation", "Decision_Margin_CSS", "Decision_Margin_All"] if metric in df_open.columns}
    return {"stage15": {"results_14_self": results_14_self, "feature_space": results_14_self["feature_space"], "active_recomputed": True}, "stage19": {"df": df_open, "stats": stats_open, "feature_space": _feature_space_from_active(), "active_recomputed": True}}


def _interaction_p_simple(df, metric):
    try:
        sub = df[["Group", "Drug", metric]].dropna().copy()
        if sub["Group"].nunique() < 2 or sub["Drug"].nunique() < 2:
            return np.nan
        y = pd.to_numeric(sub[metric], errors="coerce").to_numpy(dtype=float)
        g = sub["Group"].astype(str).str.upper().eq("SAD").astype(float).to_numpy()
        d = sub["Drug"].astype(str).str.lower().eq("oxytocin").astype(float).to_numpy()
        X = np.column_stack([np.ones(len(sub)), g, d, g * d])
        beta, _, rank, _ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ beta
        df_resid = len(y) - rank
        if df_resid <= 0:
            return np.nan
        sigma2 = float(np.sum(resid ** 2) / df_resid)
        cov = sigma2 * np.linalg.pinv(X.T @ X)
        se = float(np.sqrt(cov[3, 3]))
        return float(2 * stats.t.sf(abs(beta[3] / se), df_resid)) if se > 0 else np.nan
    except Exception:
        return np.nan


def _recompute_stage16(base):
    data_subsets = base["data_subsets"]
    n_features = np.asarray(_phase_data(data_subsets, "SAD", "Placebo", "ext")["X"]).shape[1]
    rows = []
    for group in ["SAD", "HC"]:
        mask = _as_mask(group, n_features)
        for drug in ["Placebo", "Oxytocin"]:
            ext = _phase_data(data_subsets, group, drug, "ext")
            for sub in _subject_ids(ext):
                csm = _mean_for(ext, "CS-", mask, sub=sub)
                css = _mean_for(ext, "CSS", mask, sub=sub)
                csr = _mean_for(ext, "CSR", mask, sub=sub)
                if csm is None or css is None or csr is None:
                    continue
                dist_safety = 1 - _cosine(css, csm)
                dist_threat = 1 - _cosine(csr, css)
                rows.append({"Subject": sub, "Group": group, "Drug": drug, "Dist_Safety_PV": dist_safety / max(1, int(mask.sum())), "Dist_Threat_PV": dist_threat / max(1, int(mask.sum()))})
    df = pd.DataFrame(rows)
    return {"results_21_pv": {"df": df, "p_safe": _interaction_p_simple(df, "Dist_Safety_PV"), "p_threat": _interaction_p_simple(df, "Dist_Threat_PV"), "feature_space": _feature_space_from_active(), "active_recomputed": True}, "feature_space": _feature_space_from_active(), "active_recomputed": True}


def _recompute_stage18(base):
    data_subsets = base["data_subsets"]
    n_features = np.asarray(_phase_data(data_subsets, "SAD", "Placebo", "ext")["X"]).shape[1]
    rows = []
    for group in ["SAD", "HC"]:
        mask = _as_mask(group, n_features)
        for drug in ["Placebo", "Oxytocin"]:
            rows.extend(_drift_rows_for_group(data_subsets, group, drug, mask))
    df = pd.DataFrame(rows).rename(columns={"projection": "Projection", "cosine": "Cosine"})
    stats_dict = {}
    for domain in ["Safety", "Threat", "Threat Shock Target"]:
        sub = df[df["Domain"] == domain]
        for metric in ["Cosine", "Projection"]:
            stats_dict[f"{domain}_{metric}"] = _interaction_p_simple(sub.rename(columns={metric: "value"}), "value") if metric in sub.columns else np.nan
    return {"results_22": {"df": df, "stats": stats_dict, "feature_space": _feature_space_from_active(), "active_recomputed": True}, "feature_space": _feature_space_from_active(), "active_recomputed": True}


base_inputs, base_err = _load_active_base_inputs()
if not globals().get("ACTIVE_IMPORTANCE_MASKS"):
    display(Markdown("_Active-mask recomputation skipped: run section 3 first so ACTIVE_IMPORTANCE_MASKS exists._"))
elif base_inputs is None:
    display(Markdown(f"_Active-mask recomputation skipped: {base_err}._"))
elif any(int(np.asarray(mask, dtype=bool).sum()) < 1 for mask in globals().get("ACTIVE_IMPORTANCE_MASKS", {}).values()):
    zero_counts = {group: int(np.asarray(mask, dtype=bool).sum()) for group, mask in globals().get("ACTIVE_IMPORTANCE_MASKS", {}).items()}
    display(Markdown(f"_Active-mask recomputation skipped: {globals().get('ACTIVE_IMPORTANCE_MASK_LABEL', 'ACTIVE_IMPORTANCE_MASKS')} selected too few features for recomputation. Counts: {zero_counts}._"))
else:
    try:
        _ACTIVE_BASE_INPUTS_FOR_PHASE = base_inputs
        ACTIVE_RECOMPUTED_RESULTS["stage12_topology"] = _recompute_stage12(base_inputs)
        ACTIVE_RECOMPUTED_RESULTS["stage13_drift"] = _recompute_stage13(base_inputs)
        ACTIVE_RECOMPUTED_RESULTS["stage14_trajectories"] = _recompute_stage14(base_inputs)
        decision_payloads = _recompute_stage15_and_19(base_inputs)
        ACTIVE_RECOMPUTED_RESULTS["stage15_decision"] = decision_payloads["stage15"]
        ACTIVE_RECOMPUTED_RESULTS["stage16_safety_threat"] = _recompute_stage16(base_inputs)
        ACTIVE_RECOMPUTED_RESULTS["stage18_drift_efficiency"] = _recompute_stage18(base_inputs)
        ACTIVE_RECOMPUTED_RESULTS["stage19_prob_opening"] = decision_payloads["stage19"]
        for key, payload in ACTIVE_RECOMPUTED_RESULTS.items():
            results[key] = payload
        summary_rows = []
        for key, payload in ACTIVE_RECOMPUTED_RESULTS.items():
            summary_rows.append({"stage": key, "active_recomputed": payload.get("active_recomputed", False), "feature_counts": _feature_space_counts(payload.get("feature_space", {}))})
        display(Markdown(f"**Active-mask recomputation complete:** {globals().get('ACTIVE_IMPORTANCE_MASK_LABEL', 'ACTIVE_IMPORTANCE_MASKS')}"))
        display_df(pd.DataFrame(summary_rows), rows=20)
    except Exception as exc:
        display(Markdown(f"_Active-mask recomputation failed: {type(exc).__name__}: {exc}_"))
        raise


def require_active_importance_payload(payload, label):
    if isinstance(payload, dict) and payload.get("active_recomputed"):
        display(Markdown(f"**{label}: using notebook-side recomputation with {globals().get('ACTIVE_IMPORTANCE_MASK_LABEL', 'ACTIVE_IMPORTANCE_MASKS')}.**"))
        return payload
    return payload if _payload_matches_active_importance(payload, label) else {}


## 4. Static Representational Topology
This section uses the notebook-side active-mask recomputation generated immediately above. The displayed RDMs, statistics, and figures should therefore reflect `ACTIVE_IMPORTANCE_MASKS` from section 3, not the saved downstream p < .05 payload.


In [ ]:
r12_payload = results.get("stage12_topology", {})
r12 = r12_payload.get("results_12", r12_payload) if isinstance(r12_payload, dict) else {}
RDM_CONDITIONS = r12.get("RDM_CONDITIONS") or r12_payload.get("RDM_CONDITIONS") or ["CS-", "CSS", "CSR"]
I_CS_MINUS, I_CSS, I_CSR = 0, 1, 2

def get_sig_star(p):
    if pd.isna(p): return "n/a"
    if p < 0.001: return "***"
    if p < 0.01: return "**"
    if p < 0.05: return "*"
    return "ns"

def extract_metrics(rdms):
    arr = np.asarray(rdms, dtype=float)
    return arr[:, I_CSR, I_CSS], arr[:, I_CSS, I_CS_MINUS]

def _stats_tuple(key, sad_vec, hc_vec):
    val = r12.get(key)
    if isinstance(val, (list, tuple, np.ndarray)) and len(val) >= 2:
        return float(val[0]), float(val[1])
    t, pval = stats.ttest_ind(sad_vec, hc_vec, equal_var=False, nan_policy='omit')
    return float(t), float(pval)

def _one_sample_p(prefix, key, vec):
    d = r12.get(prefix, {}) if isinstance(r12.get(prefix, {}), dict) else {}
    if key in d:
        return float(d[key])
    return float(stats.ttest_1samp(vec, 0, nan_policy='omit').pvalue)

def _print_metric_report(label, vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc, stat_a, stat_b, one_prefix):
    print("\n" + "="*110)
    print(label)
    print("="*110)
    rows = []
    for metric_label, sad_vec, hc_vec, stat_key, stat_pair in [
        ("Threat vs Safety", vec_a_sad, vec_a_hc, "a", stat_a),
        ("Safety vs Backgr", vec_b_sad, vec_b_hc, "b", stat_b),
    ]:
        p_sad = _one_sample_p(one_prefix, f"p_{stat_key}_sad", sad_vec)
        p_hc = _one_sample_p(one_prefix, f"p_{stat_key}_hc", hc_vec)
        t_diff, p_diff = stat_pair
        rows.extend([
            {"Metric": metric_label, "Group": "SAD", "Mean": np.nanmean(sad_vec), "p_vs_0": p_sad, "GroupDiff_t": t_diff, "GroupDiff_p": p_diff},
            {"Metric": metric_label, "Group": "HC", "Mean": np.nanmean(hc_vec), "p_vs_0": p_hc, "GroupDiff_t": np.nan, "GroupDiff_p": np.nan},
        ])
        print(f"{metric_label:<24} | SAD mean={np.nanmean(sad_vec):.6f}, p(vs0)={p_sad:.4f} | HC mean={np.nanmean(hc_vec):.6f}, p(vs0)={p_hc:.4f} | Group t={t_diff:.3f}, p={p_diff:.4f}")
    display_df(pd.DataFrame(rows), rows=10)

def plot_topology(rdms_sad, rdms_hc, vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc, p_a, p_b, p_a_sad_0, p_a_hc_0, p_b_sad_0, p_b_hc_0, title_suffix, filename):
    sns.set_context("poster")
    fig = plt.figure(figsize=(24, 8))
    gs = fig.add_gridspec(1, 3)
    ax1 = fig.add_subplot(gs[0, 0])
    sns.heatmap(np.nanmean(rdms_sad, axis=0), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1.2, xticklabels=RDM_CONDITIONS, yticklabels=RDM_CONDITIONS, ax=ax1, cbar=False)
    ax1.set_title(f"SAD Topology ({title_suffix})\n(n={len(rdms_sad)})")
    ax2 = fig.add_subplot(gs[0, 1])
    sns.heatmap(np.nanmean(rdms_hc, axis=0), annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1.2, xticklabels=RDM_CONDITIONS, yticklabels=RDM_CONDITIONS, ax=ax2)
    ax2.set_title(f"HC Topology ({title_suffix})\n(n={len(rdms_hc)})")
    ax3 = fig.add_subplot(gs[0, 2])
    df_res = pd.DataFrame({
        'Group': ['SAD'] * len(vec_a_sad) + ['HC'] * len(vec_a_hc) + ['SAD'] * len(vec_b_sad) + ['HC'] * len(vec_b_hc),
        'Distance': np.concatenate([vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc]),
        'Metric': ['A: Threat Dist'] * len(vec_a_sad) + ['A: Threat Dist'] * len(vec_a_hc) + ['B: Safety Dist'] * len(vec_b_sad) + ['B: Safety Dist'] * len(vec_b_hc),
    })
    sns.violinplot(data=df_res, x='Metric', y='Distance', hue='Group', split=True, inner='quartile', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=ax3)
    ax3.set_title(f"Topological Metrics (Centroid | {title_suffix})")
    ax3.set_ylabel("Crossnobis Distance")
    y_max = df_res['Distance'].max()
    if p_a < 0.05:
        ax3.text(0, y_max + 0.05, f'* (p={p_a:.3f})', ha='center', fontsize=18)
    if p_b < 0.05:
        ax3.text(1, y_max + 0.05, f'* (p={p_b:.3f})', ha='center', fontsize=18)
    ax3.text(-0.2, -0.15, f"SAD: {get_sig_star(p_a_sad_0)}", transform=ax3.get_xaxis_transform(), ha='center', fontsize=14, color='#c44e52')
    ax3.text(0.2, -0.15, f"HC: {get_sig_star(p_a_hc_0)}", transform=ax3.get_xaxis_transform(), ha='center', fontsize=14, color='#4c72b0')
    ax3.text(0.8, -0.15, f"SAD: {get_sig_star(p_b_sad_0)}", transform=ax3.get_xaxis_transform(), ha='center', fontsize=14, color='#c44e52')
    ax3.text(1.2, -0.15, f"HC: {get_sig_star(p_b_hc_0)}", transform=ax3.get_xaxis_transform(), ha='center', fontsize=14, color='#4c72b0')
    save_current_fig(filename)
    plt.show()

display_df(flatten_result_dict(r12), rows=80)
plot_specs = [
    ("raw", "rdms_sad_raw", "rdms_hc_raw", "metric_a_stats_raw", "metric_b_stats_raw", "one_sample_stats_raw", "Active p<.01 Recomputed\n (Raw)", "memoryfearnetwork_topology_raw_active_pLess0.01.png"),
    ("z", "rdms_sad_z", "rdms_hc_z", "metric_a_stats_z", "metric_b_stats_z", "one_sample_stats_z", "Active p<.01 Recomputed\n (Z-Scored)", "memoryfearnetwork_topology_z_active_pLess0.01.png"),
    ("raw_pv", "rdms_sad_raw_pv", "rdms_hc_raw_pv", "metric_a_stats_raw_pv", "metric_b_stats_raw_pv", "one_sample_stats_raw_pv", "Active p<.01 Recomputed\n (Raw, Per-Voxel)", "memoryfearnetwork_topology_raw_pv_active_pLess0.01.png"),
]
for label, sad_key, hc_key, stat_a_key, stat_b_key, one_prefix, title_suffix, filename in plot_specs:
    if sad_key in r12 and hc_key in r12:
        rdms_sad = np.asarray(r12[sad_key], dtype=float)
        rdms_hc = np.asarray(r12[hc_key], dtype=float)
        vec_a_sad, vec_b_sad = extract_metrics(rdms_sad)
        vec_a_hc, vec_b_hc = extract_metrics(rdms_hc)
        stat_a = _stats_tuple(stat_a_key, vec_a_sad, vec_a_hc)
        stat_b = _stats_tuple(stat_b_key, vec_b_sad, vec_b_hc)
        _print_metric_report(f"TOPOLOGY REPORT: {label}", vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc, stat_a, stat_b, one_prefix)
        p_a_sad = _one_sample_p(one_prefix, "p_a_sad", vec_a_sad)
        p_a_hc = _one_sample_p(one_prefix, "p_a_hc", vec_a_hc)
        p_b_sad = _one_sample_p(one_prefix, "p_b_sad", vec_b_sad)
        p_b_hc = _one_sample_p(one_prefix, "p_b_hc", vec_b_hc)
        plot_topology(rdms_sad, rdms_hc, vec_a_sad, vec_a_hc, vec_b_sad, vec_b_hc, stat_a[1], stat_b[1], p_a_sad, p_a_hc, p_b_sad, p_b_hc, title_suffix, filename)


# Shock is intentionally not plotted as a fourth RDM condition. Its global evoked
# amplitude can dominate crossnobis distance, so section 4 reports a separate
# residualized Shock-anchor sensitivity instead.
def _shock_anchor_payload(r12_dict):
    if not isinstance(r12_dict, dict):
        return pd.DataFrame(), {}
    anchor = r12_dict.get("shock_anchor_results", {})
    df = r12_dict.get("shock_anchor_df")
    stats_dict = r12_dict.get("shock_anchor_stats")
    if df is None and isinstance(anchor, dict):
        df = anchor.get("df")
    if stats_dict is None and isinstance(anchor, dict):
        stats_dict = anchor.get("stats", {})
    if not isinstance(df, pd.DataFrame):
        df = pd.DataFrame()
    if not isinstance(stats_dict, dict):
        stats_dict = {}
    return df, stats_dict

shock_anchor_df, shock_anchor_stats = _shock_anchor_payload(r12)
display(Markdown("**Shock-anchor sensitivity (residualized; not part of main RDM)**"))
display(Markdown("CSS and CSR are projected onto each subject's Shock-minus-CS- axis after removing each trial's feature-wise mean. This is a sensitivity analysis, not the primary topology RDM."))
if isinstance(shock_anchor_df, pd.DataFrame) and not shock_anchor_df.empty:
    show_cols = [c for c in ["Subject", "Group", "Projection_CSS", "Projection_CSR", "Projection_CSR_minus_CSS", "Cosine_CSS", "Cosine_CSR", "Cosine_CSR_minus_CSS", "Shock_Axis_Norm", "n_shock_trials"] if c in shock_anchor_df.columns]
    display_df(shock_anchor_df[show_cols], rows=30)
    rows = []
    for metric, info in shock_anchor_stats.items():
        if not isinstance(info, dict):
            continue
        means = info.get("means", {}) if isinstance(info.get("means", {}), dict) else {}
        group_stats = info.get("group_stats", (np.nan, np.nan))
        try:
            t_val, p_val = group_stats[:2]
        except Exception:
            t_val, p_val = np.nan, np.nan
        rows.append({"Metric": metric, "SAD_mean": means.get("SAD", np.nan), "HC_mean": means.get("HC", np.nan), "t_group": t_val, "p_group": p_val})
    if rows:
        stats_df = pd.DataFrame(rows)
        display_df(stats_df, rows=30)
    plot_metrics = [c for c in ["Projection_CSS", "Projection_CSR", "Projection_CSR_minus_CSS"] if c in shock_anchor_df.columns]
    if plot_metrics:
        long_df = shock_anchor_df.melt(id_vars=["Subject", "Group"], value_vars=plot_metrics, var_name="Metric", value_name="Projection")
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.boxplot(data=long_df, x="Metric", y="Projection", hue="Group", palette={"SAD": "#c44e52", "HC": "#4c72b0"}, ax=ax)
        sns.stripplot(data=long_df, x="Metric", y="Projection", hue="Group", dodge=True, color="black", alpha=0.35, size=3, ax=ax)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles[:2], labels[:2], title="Group", loc="best")
        ax.axhline(0, color="gray", ls="--", lw=1)
        ax.set_title("Residualized Shock-Anchor Projection")
        ax.set_ylabel("Projection onto Shock-minus-CS- axis")
        ax.tick_params(axis="x", rotation=15)
        plt.tight_layout()
        save_current_fig(f"{PIPELINE_NAME.lower()}_shock_anchor_projection.png")
        plt.show()
else:
    display(Markdown("_No Shock-anchor projection results found. Re-run stage 12 after the Shock-anchor update; old Shock-inclusive RDM fields are no longer used._"))


## 5. Dynamic Representational Drift
This section displays drift statistics and the drift figures saved by the script, then regenerates compact numeric summaries when possible.


In [ ]:
r13 = results.get("stage13_drift", {})
res13 = r13.get("results_13", r13) if isinstance(r13, dict) else {}
df_plot = res13.get("df_plot") if isinstance(res13, dict) else None
display_df(flatten_result_dict(res13), rows=60)
if isinstance(df_plot, pd.DataFrame) and not df_plot.empty:
    print(f"\n[Step 3] Analyzing {len(df_plot)} subject vectors...")
    display_df(df_plot, rows=25)
    sns.set_context("poster")
    fig, axes = plt.subplots(1, 3, figsize=(26, 8), gridspec_kw={'height_ratios': [1]})
    sns.barplot(data=df_plot, x='Condition', y='projection', hue='Group', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=axes[0], capsize=.1)
    axes[0].axhline(0, color='k', ls='--')
    axes[0].set_title("Neural Plasticity Magnitude\n(Projection)")
    axes[0].set_ylabel("Plasticity (au)")
    sns.barplot(data=df_plot, x='Condition', y='cosine', hue='Group', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=axes[1], capsize=.1)
    axes[1].axhline(0, color='k', ls='--')
    axes[1].set_title("Representational Fidelity\n(Cosine Similarity)")
    axes[1].set_ylabel("Fidelity (cos)")
    sns.scatterplot(data=df_plot, x='init_dist', y='projection', hue='Group', style='Condition', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=axes[2], s=150, alpha=0.8, edgecolor='black')
    axes[2].axhline(0, color='k', ls='--')
    axes[2].set_title("Plasticity vs. Initial Distance\n(Voxel Space)")
    axes[2].set_ylabel("Plasticity (au)")
    axes[2].grid(True, linestyle=':', alpha=0.5)
    print("\n--- Statistical Summary (p-values) ---")
    stat_rows = []
    for met in ['projection', 'cosine']:
        for cond in ['Safety', 'Threat']:
            d_s = df_plot[(df_plot['Condition'] == cond) & (df_plot['Group'] == 'SAD')][met].dropna()
            d_h = df_plot[(df_plot['Condition'] == cond) & (df_plot['Group'] == 'HC')][met].dropna()
            if len(d_s) > 1 and len(d_h) > 1:
                t, pval = stats.ttest_ind(d_s, d_h)
                sig = "*" if pval < 0.05 else "ns"
                print(f"  > Group Diff: {cond} {met} t={t:.3f}, p={pval:.4f} {sig}")
                stat_rows.append({'Condition': cond, 'Metric': met, 't': t, 'p': pval, 'sig': sig, 'mean_SAD': d_s.mean(), 'mean_HC': d_h.mean()})
    display_df(pd.DataFrame(stat_rows), rows=20)
    save_current_fig("memoryfearnetwork_stage11_drift_plots_row_pLess0.01.png")
    plt.show()
else:
    display(Markdown("_No dynamic drift df_plot found._"))


## 6. Single-Trial Safety and Threat Trajectories
Safety and threat are plotted separately because they may use different trajectory scores. Trial-wise SAD versus HC Welch tests are reported when trial-level data are available.


In [ ]:

r14 = results.get("stage14_trajectories", {})
if not isinstance(r14, dict) or not r14:
    r14 = results.get("stage13_DynamicRepresentationalDrift", {})
res132 = r14.get("results_13_2", r14) if isinstance(r14, dict) else {}
__ALLOW_DIRECT_TRAJECTORY_CACHE__ = False
if __ALLOW_DIRECT_TRAJECTORY_CACHE__ and not (isinstance(res132, dict) and "data_threat_shock" in res132):
    for _traj_path in [CHECKPOINT_DIR / "cell_12_trajectories.joblib", CHECKPOINT_DIR / "cell_14.joblib", INTERMEDIATE_DIR / "stage14_trajectories.joblib"]:
        if _traj_path.exists():
            try:
                _traj_payload = joblib.load(_traj_path)
            except Exception:
                continue
            _traj_res = _traj_payload.get("results_13_2", _traj_payload) if isinstance(_traj_payload, dict) else {}
            if isinstance(_traj_res, dict) and "data_threat_shock" in _traj_res:
                display(Markdown(f"_Loaded trajectory results directly from `{_traj_path}`._"))
                res132 = _traj_res
                break
elif not __ALLOW_DIRECT_TRAJECTORY_CACHE__ and not (isinstance(res132, dict) and res132.get("active_recomputed")):
    display(Markdown("_Variant notebook: section 6 uses notebook-side active-mask recomputation. Run section 3 first; cached trajectory joblibs are intentionally not used here._"))

stats_safe = res132.get("stats_safe") if isinstance(res132, dict) else None
stats_threat = res132.get("stats_threat") if isinstance(res132, dict) else None
stats_threat_shock = res132.get("stats_threat_shock") if isinstance(res132, dict) else None
df_safe = res132.get("data_safe") if isinstance(res132, dict) else None
df_threat = res132.get("data_threat") if isinstance(res132, dict) else None
df_threat_shock = res132.get("data_threat_shock") if isinstance(res132, dict) else None

display(Markdown("**Safety trajectory statistics**")); display_df(stats_safe, rows=30)
display(Markdown("**Threat maintenance trajectory statistics**")); display_df(stats_threat, rows=30)
display(Markdown("**Threat shock-target trajectory statistics**")); display_df(stats_threat_shock, rows=30)

def _format_p_for_plot(p):
    if pd.isna(p):
        return "NA"
    p = float(p)
    if p < 0.001:
        return "<.001"
    return f"={p:.3f}".replace("0.", ".")

def _annotate_sig_group_diffs(ax, data_df, stats_df):
    if not isinstance(data_df, pd.DataFrame) or data_df.empty or not isinstance(stats_df, pd.DataFrame) or stats_df.empty:
        return
    if "Trial" not in stats_df.columns:
        return
    use_col = None
    label_prefix = "p"
    if "Diff_p_fdr" in stats_df.columns and (pd.to_numeric(stats_df["Diff_p_fdr"], errors="coerce") < 0.05).any():
        use_col = "Diff_p_fdr"
        label_prefix = "q"
    elif "Diff_p" in stats_df.columns:
        use_col = "Diff_p"
    if use_col is None:
        return
    sig = stats_df.loc[pd.to_numeric(stats_df[use_col], errors="coerce") < 0.05].copy()
    if sig.empty:
        return
    base_y = 1.72
    step = 0.09
    for i, (_, row) in enumerate(sig.iterrows()):
        trial = float(row["Trial"])
        pval = float(row[use_col])
        y = min(base_y + (i % 3) * step, 1.95)
        ax.plot([trial - 0.22, trial + 0.22], [y - 0.045, y - 0.045], color="black", lw=1.6, clip_on=True)
        ax.text(trial, y, f"{label_prefix}{_format_p_for_plot(pval)}", ha="center", va="bottom", fontsize=14, color="black", clip_on=True)

trajectory_panels = [
    ("A. Safety Trajectory\n(Target = CS-)", df_safe, stats_safe, "o", "#2ca02c", "Target (CS-)", "Start (Fear)"),
    ("B. Threat Maintenance\n(Target = Reinstated CSR)", df_threat, stats_threat, "s", "#d62728", "Target (Reinstated CSR)", "Start (Ext Early)"),
    ("C. Threat Acquisition\n(Target = Shock/US)", df_threat_shock, stats_threat_shock, "^", "#9467bd", "Target (Shock/US)", "Start (Ext Early)"),
]
available_panels = [(title, df, stats_df, marker, target_color, target_label, start_label) for title, df, stats_df, marker, target_color, target_label, start_label in trajectory_panels if isinstance(df, pd.DataFrame) and not df.empty]
if available_panels:
    sns.set_context("poster")
    fig, axes = plt.subplots(1, len(available_panels), figsize=(11 * len(available_panels), 9), sharey=True, squeeze=False)
    axes = axes.ravel()
    for ax, (title, df_plot_panel, stats_panel, marker, target_color, target_label, start_label) in zip(axes, available_panels):
        sns.lineplot(data=df_plot_panel, x="trial", y="score", hue="Group", palette={"SAD": "#c44e52", "HC": "#4c72b0"}, lw=3, marker=marker, err_style="band", ax=ax)
        ax.set_title(title)
        ax.set_xlabel("Trial (Block Size: 1)")
        ax.set_ylabel("Similarity Score (0=Start, 1=Target)")
        ax.axhline(0, color="gray", ls="--", label=start_label)
        ax.axhline(1, color=target_color, ls="-", lw=2, label=target_label)
        _annotate_sig_group_diffs(ax, df_plot_panel, stats_panel)
        ax.set_ylim(-1, 2)
        ax.legend(loc="upper left")
    save_current_fig(f"{PIPELINE_NAME.lower()}_stage14_trajectories_three_panel.png")
    plt.show()
else:
    display(Markdown("_No trajectory dataframes found. For variant notebooks, run section 3 first so trajectories are recomputed from ACTIVE_IMPORTANCE_MASKS._"))


## 6b. Trial-Wise SCR-Neural Coupling
This section correlates trial-wise SCR with single-trial neural trajectory scores when both saved outputs are available. Safety uses CSS SCR with safety trajectory score; threat uses CSR SCR with threat trajectory score.


In [ ]:
r14 = results.get("stage14_trajectories", {})
res132 = r14.get("results_13_2", r14) if isinstance(r14, dict) else {}
df_safe = res132.get('data_safe') if isinstance(res132, dict) else None
df_threat = res132.get('data_threat') if isinstance(res132, dict) else None
df_threat_shock = res132.get('data_threat_shock') if isinstance(res132, dict) else None
r23 = results.get("stage23_clinical_scores", {})
r24 = results.get("stage24_neural_indices", {})
r26 = results.get("stage26_master_merge", {})

def _first_df_from_payload(payload, keys):
    if not isinstance(payload, dict):
        return None
    for key in keys:
        val = payload.get(key)
        if isinstance(val, pd.DataFrame):
            return val
    return None

def _first_nonempty_df(*dfs):
    for df in dfs:
        if isinstance(df, pd.DataFrame):
            return df
    return None


df_scr_trials = _first_nonempty_df(
    _first_df_from_payload(r24, ["df_scr_trials"]),
    _first_df_from_payload(r23, ["df_scr_trials"]),
)
meta_for_scr = _first_nonempty_df(
    _first_df_from_payload(r23, ["meta"]),
    _first_df_from_payload(r24, ["meta"]),
    _first_df_from_payload(r26, ["meta"]),
)

def _normalize_sub_id(s):
    return str(s).replace("sub-", "").strip()

def _attach_meta(df):
    if not isinstance(df, pd.DataFrame) or df.empty or not isinstance(meta_for_scr, pd.DataFrame):
        return df
    meta = meta_for_scr.copy()
    sub_col = "subject_id" if "subject_id" in meta.columns else "sub_ID" if "sub_ID" in meta.columns else None
    if sub_col is None:
        return df
    keep = [sub_col] + [c for c in ["Group", "Drug", "drug_condition", "Gender", "demo_age"] if c in meta.columns]
    meta = meta[keep].drop_duplicates(sub_col).copy()
    meta["sub_ID"] = meta[sub_col].map(_normalize_sub_id)
    out = df.copy()
    out["sub_ID"] = out["sub_ID"].map(_normalize_sub_id)
    merge_cols = ["sub_ID"] + [c for c in meta.columns if c not in {sub_col, "sub_ID"}]
    out = out.merge(meta[merge_cols], on="sub_ID", how="left", suffixes=("", "_meta"))
    if "Group_meta" in out.columns and "Group" in out.columns:
        out["Group"] = out["Group"].fillna(out["Group_meta"])
        out = out.drop(columns=["Group_meta"])
    return out

def _compute_trialwise_scr_neural(neural_df, scr_df, scr_condition, domain):
    if not isinstance(neural_df, pd.DataFrame) or neural_df.empty or not isinstance(scr_df, pd.DataFrame) or scr_df.empty:
        return pd.DataFrame(), pd.DataFrame()
    required_neural = {"sub", "trial", "score"}
    required_scr = {"sub_ID", "SCR_Condition", "condition_trial", "SCR_Anticipatory"}
    if not required_neural.issubset(neural_df.columns) or not required_scr.issubset(scr_df.columns):
        return pd.DataFrame(), pd.DataFrame()
    neural = neural_df.rename(columns={"sub": "sub_ID", "trial": "condition_trial", "score": "Neural_Trajectory_Score"}).copy()
    neural["sub_ID"] = neural["sub_ID"].map(_normalize_sub_id)
    neural["condition_trial"] = pd.to_numeric(neural["condition_trial"], errors="coerce")
    keep_cols = ["sub_ID", "condition_trial", "Neural_Trajectory_Score"] + [c for c in ["Group", "Condition"] if c in neural.columns]
    neural = neural[keep_cols]
    scr = scr_df[scr_df["SCR_Condition"].astype(str).eq(scr_condition)].copy()
    scr["sub_ID"] = scr["sub_ID"].map(_normalize_sub_id)
    scr["condition_trial"] = pd.to_numeric(scr["condition_trial"], errors="coerce")
    scr["SCR_Anticipatory"] = pd.to_numeric(scr["SCR_Anticipatory"], errors="coerce")
    merged = neural.merge(scr[["sub_ID", "condition_trial", "SCR_Anticipatory", "SCR_Trial"]], on=["sub_ID", "condition_trial"], how="inner")
    merged["Domain"] = domain
    merged["SCR_Condition"] = scr_condition
    merged = _attach_meta(merged)
    rows = []
    for sub_id, sub_df in merged.groupby("sub_ID"):
        valid = sub_df[["Neural_Trajectory_Score", "SCR_Anticipatory"]].dropna()
        r_val, p_val = np.nan, np.nan
        if len(valid) >= 3 and valid["Neural_Trajectory_Score"].nunique() > 1 and valid["SCR_Anticipatory"].nunique() > 1:
            r_val, p_val = stats.pearsonr(valid["Neural_Trajectory_Score"], valid["SCR_Anticipatory"])
        group = sub_df["Group"].dropna().iloc[0] if "Group" in sub_df.columns and sub_df["Group"].notna().any() else np.nan
        drug = sub_df["Drug"].dropna().iloc[0] if "Drug" in sub_df.columns and sub_df["Drug"].notna().any() else np.nan
        rows.append({"sub_ID": sub_id, "Group": group, "Drug": drug, "Domain": domain, "SCR_Condition": scr_condition, "r_neural_scr": r_val, "p_neural_scr": p_val, "n_trials": int(len(valid))})
    return merged, pd.DataFrame(rows)

merged_frames = []
coupling_frames = []
for domain, neural_df, scr_condition in [("Safety", df_safe, "CSS"), ("Threat", df_threat, "CSR"), ("Threat Shock Target", df_threat_shock, "CSR")]:
    merged, coupling = _compute_trialwise_scr_neural(neural_df, df_scr_trials, scr_condition, domain)
    if not merged.empty:
        merged_frames.append(merged)
    if not coupling.empty:
        coupling_frames.append(coupling)

if coupling_frames:
    trial_scr_neural = pd.concat(merged_frames, ignore_index=True) if merged_frames else pd.DataFrame()
    subject_scr_neural_coupling = pd.concat(coupling_frames, ignore_index=True)
    display(Markdown("**Subject-level trial-wise SCR-neural coupling**"))
    display_df(subject_scr_neural_coupling, rows=80)

    def _p_to_stars(p):
        try:
            p = float(p)
        except Exception:
            return ""
        if not np.isfinite(p):
            return ""
        if p < 0.001:
            return "***"
        if p < 0.01:
            return "**"
        if p < 0.05:
            return "*"
        if p < 0.10:
            return "+"
        return ""

    stats_rows = []
    for domain, sub_df in subject_scr_neural_coupling.groupby("Domain"):
        sad = pd.to_numeric(sub_df.loc[sub_df["Group"].astype(str).str.upper().eq("SAD"), "r_neural_scr"], errors="coerce").dropna()
        hc = pd.to_numeric(sub_df.loc[sub_df["Group"].astype(str).str.upper().eq("HC"), "r_neural_scr"], errors="coerce").dropna()
        if len(sad) >= 2 and len(hc) >= 2:
            t_val, p_val = stats.ttest_ind(sad, hc, equal_var=False, nan_policy="omit")
        else:
            t_val, p_val = np.nan, np.nan
        stats_rows.append({"Domain": domain, "SAD_n": len(sad), "HC_n": len(hc), "SAD_mean_r": sad.mean() if len(sad) else np.nan, "HC_mean_r": hc.mean() if len(hc) else np.nan, "Welch_t": t_val, "p": p_val, "sig": _p_to_stars(p_val)})
    stats_table = pd.DataFrame(stats_rows)
    display(Markdown("**SAD vs HC tests of subject-level coupling**"))
    display_df(stats_table, rows=20)

    display(Markdown("**Primary trial-wise mixed-effects model**"))
    display(Markdown("Full-sample model: z(SCR) ~ z(neural trajectory score) * Group * Domain + Drug + trial + subject random effects. The code first tries a subject-level random neural-score slope and falls back to a random intercept if needed."))

    def _zscore_series(s):
        x = pd.to_numeric(s, errors="coerce")
        sd = x.std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            return x * np.nan
        return (x - x.mean()) / sd

    def _mixedlm_fixed_effects_table(fit, label, random_effects):
        conf = fit.conf_int()
        rows = []
        for term in fit.params.index:
            if term == "Group Var" or term.endswith(" Var") or " Cov" in term:
                continue
            rows.append({
                "model": label,
                "random_effects": random_effects,
                "term": term,
                "beta": fit.params.get(term, np.nan),
                "se": fit.bse.get(term, np.nan) if hasattr(fit, "bse") else np.nan,
                "z": fit.tvalues.get(term, np.nan) if hasattr(fit, "tvalues") else np.nan,
                "p": fit.pvalues.get(term, np.nan) if hasattr(fit, "pvalues") else np.nan,
                "sig": _p_to_stars(fit.pvalues.get(term, np.nan) if hasattr(fit, "pvalues") else np.nan),
                "ci_low": conf.loc[term, 0] if term in conf.index else np.nan,
                "ci_high": conf.loc[term, 1] if term in conf.index else np.nan,
                "n_trials": int(fit.nobs),
                "n_subjects": int(len(np.unique(fit.model.groups))),
            })
        return pd.DataFrame(rows)

    def _fit_trialwise_mixedlm(model_df, label, formula):
        try:
            import statsmodels.formula.api as smf
        except Exception as exc:
            return pd.DataFrame(), f"statsmodels unavailable for {label}: {exc}"
        dfm = model_df.copy()
        dfm = dfm.dropna(subset=["Neural_Trajectory_Score", "SCR_Anticipatory", "sub_ID", "Group", "Domain"])
        if dfm.empty or dfm["sub_ID"].nunique() < 3:
            return pd.DataFrame(), f"Not enough subjects for {label}."
        dfm["Neural_Z"] = _zscore_series(dfm["Neural_Trajectory_Score"])
        dfm["SCR_Z"] = _zscore_series(dfm["SCR_Anticipatory"])
        dfm["Trial_Z"] = _zscore_series(dfm["condition_trial"]) if "condition_trial" in dfm.columns else 0.0
        dfm["Group"] = dfm["Group"].astype(str).str.upper()
        dfm["Domain"] = dfm["Domain"].astype(str)
        if "Drug" not in dfm.columns:
            dfm["Drug"] = "Unknown"
        dfm["Drug"] = dfm["Drug"].fillna("Unknown").astype(str)
        dfm = dfm.dropna(subset=["Neural_Z", "SCR_Z", "Trial_Z"])
        if dfm.empty or dfm["SCR_Z"].nunique() < 2 or dfm["Neural_Z"].nunique() < 2:
            return pd.DataFrame(), f"Not enough variance for {label}."
        attempts = [("random intercept + neural-score slope", "~Neural_Z"), ("random intercept", "1")]
        last_error = None
        for random_label, re_formula in attempts:
            try:
                fit = smf.mixedlm(formula, data=dfm, groups=dfm["sub_ID"], re_formula=re_formula).fit(reml=False, method="lbfgs", maxiter=500, disp=False)
                return _mixedlm_fixed_effects_table(fit, label, random_label), ""
            except Exception as exc:
                last_error = exc
        return pd.DataFrame(), f"MixedLM failed for {label}: {last_error}"

    mixed_tables = []
    mixed_messages = []
    if not trial_scr_neural.empty:
        combined_formula = "SCR_Z ~ Neural_Z * C(Group) * C(Domain) + C(Drug) + Trial_Z"
        table, msg = _fit_trialwise_mixedlm(trial_scr_neural, "Full sample: SCR outcome", combined_formula)
        if not table.empty:
            mixed_tables.append(table)
        if msg:
            mixed_messages.append(msg)
        for domain, domain_df in trial_scr_neural.groupby("Domain"):
            domain_formula = "SCR_Z ~ Neural_Z * C(Group) + C(Drug) + Trial_Z"
            table, msg = _fit_trialwise_mixedlm(domain_df, f"{domain} only: SCR outcome", domain_formula)
            if not table.empty:
                mixed_tables.append(table)
            if msg:
                mixed_messages.append(msg)
    if mixed_tables:
        mixed_effects_table = pd.concat(mixed_tables, ignore_index=True)
        display_df(mixed_effects_table, rows=120)
        sig_mixed_effects = mixed_effects_table[pd.to_numeric(mixed_effects_table["p"], errors="coerce") < 0.05].copy()
        display(Markdown("**Significant mixed-effects terms (p < .05)**"))
        if sig_mixed_effects.empty:
            display(Markdown("_No mixed-effects terms reached p < .05._"))
        else:
            display_df(sig_mixed_effects, rows=80)
    else:
        mixed_effects_table = pd.DataFrame()
        display(Markdown("_Mixed-effects model could not be fit._"))
    for msg in mixed_messages:
        display(Markdown(f"_Mixed model note: {msg}_"))

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), squeeze=False)
    ax0, ax1 = axes.ravel()
    sns.boxplot(data=subject_scr_neural_coupling, x="Domain", y="r_neural_scr", hue="Group", palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, ax=ax0, fliersize=0)
    sns.stripplot(data=subject_scr_neural_coupling, x="Domain", y="r_neural_scr", hue="Group", dodge=True, alpha=0.55, color="black", ax=ax0, legend=False)
    ax0.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax0.set_title("Subject-level SCR-neural coupling")
    ax0.set_ylabel("Within-subject r")
    ax0.set_xlabel("")
    if ax0.get_legend() is not None:
        ax0.legend(title="Group", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, borderaxespad=0)
    if not stats_table.empty and "sig" in stats_table.columns:
        y_min, y_max = ax0.get_ylim()
        y_text = y_max - 0.06 * (y_max - y_min)
        domain_order = [tick.get_text() for tick in ax0.get_xticklabels()]
        for _, row in stats_table.iterrows():
            sig = row.get("sig", "")
            domain = str(row.get("Domain", ""))
            if sig and domain in domain_order:
                ax0.text(domain_order.index(domain), y_text, sig, ha="center", va="top", fontsize=14, fontweight="bold", color="black")
    if not trial_scr_neural.empty:
        sns.scatterplot(data=trial_scr_neural, x="SCR_Anticipatory", y="Neural_Trajectory_Score", hue="Group", style="Domain", alpha=0.35, ax=ax1)
        for group, color in [("SAD", '#c44e52'), ("HC", '#4c72b0')]:
            gdf = trial_scr_neural[trial_scr_neural["Group"].astype(str).str.upper().eq(group)]
            if len(gdf) > 5:
                sns.regplot(data=gdf, x="SCR_Anticipatory", y="Neural_Trajectory_Score", scatter=False, ax=ax1, color=color, label=f"{group} fit")
        ax1.set_title("Descriptive trial-level SCR vs neural trajectory score")
        ax1.set_xlabel("Anticipatory SCR")
        ax1.set_ylabel("Neural trajectory score")
        handles, labels = ax1.get_legend_handles_labels()
        dedup = {}
        for handle, label in zip(handles, labels):
            if label and not str(label).startswith("_") and label not in dedup:
                dedup[label] = handle
        if dedup:
            ax1.legend(list(dedup.values()), list(dedup.keys()), title="Group / domain", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, borderaxespad=0)
    else:
        ax1.axis("off")
    fig.tight_layout(w_pad=4.0)
    save_current_fig(f"{PIPELINE_NAME.lower()}_trialwise_scr_neural_coupling.png")
    plt.show()
    if not trial_scr_neural.empty:
        domains = [d for d in ["Safety", "Threat", "Threat Shock Target"] if d in set(trial_scr_neural["Domain"].dropna().astype(str))]
        if domains:
            fig_corr, axes_corr = plt.subplots(1, len(domains), figsize=(6.2 * len(domains), 4.8), squeeze=False)
            for ax_corr, domain in zip(axes_corr.ravel(), domains):
                ddf = trial_scr_neural[trial_scr_neural["Domain"].astype(str).eq(domain)].copy()
                ddf["SCR_Anticipatory"] = pd.to_numeric(ddf["SCR_Anticipatory"], errors="coerce")
                ddf["Neural_Trajectory_Score"] = pd.to_numeric(ddf["Neural_Trajectory_Score"], errors="coerce")
                ddf = ddf.dropna(subset=["SCR_Anticipatory", "Neural_Trajectory_Score"])
                if len(ddf) >= 3:
                    r_all, p_all = stats.pearsonr(ddf["SCR_Anticipatory"], ddf["Neural_Trajectory_Score"])
                else:
                    r_all, p_all = np.nan, np.nan
                corr_sig = _p_to_stars(p_all)
                sns.scatterplot(data=ddf, x="SCR_Anticipatory", y="Neural_Trajectory_Score", hue="Group", palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, alpha=0.35, ax=ax_corr)
                if len(ddf) >= 3:
                    sns.regplot(data=ddf, x="SCR_Anticipatory", y="Neural_Trajectory_Score", scatter=False, ax=ax_corr, color="black", line_kws={"linewidth": 2})
                ax_corr.set_title(f"Descriptive {domain}: SCR-neural correlation {corr_sig}\nr={r_all:.3f}, p={p_all:.3g}, n={len(ddf)}")
                ax_corr.set_xlabel("Anticipatory SCR")
                ax_corr.set_ylabel("Neural trajectory score")
                if ax_corr.get_legend() is not None:
                    ax_corr.legend(title="Group", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, borderaxespad=0)
            fig_corr.tight_layout(w_pad=4.0)
            save_current_fig(f"{PIPELINE_NAME.lower()}_trialwise_scr_neural_direct_correlation.png")
            plt.show()
    if not trial_scr_neural.empty:
        display(Markdown("**Matched trial-level rows used for coupling**"))
        display_df(trial_scr_neural, rows=40)
else:
    display(Markdown("_Trial-wise SCR-neural coupling could not be computed: missing `df_scr_trials` or trajectory data (`data_safe` / `data_threat` / `data_threat_shock`)._"))


## 6c. Placebo-Only Trial-Wise Mixed-Effects Models
This cell repeats the trial-wise mixed-effects test within the placebo arm only, including combined safety/threat and Safety-only / Threat-only follow-up models.


In [ ]:
display(Markdown("**Placebo-only trial-wise mixed-effects model**"))
display(Markdown("Placebo-only model: z(SCR) ~ z(neural trajectory score) * Group * Domain + trial + subject random effects. Safety-only and Threat-only follow-ups use z(SCR) ~ z(neural trajectory score) * Group + trial + subject random effects."))

placebo_mixed_tables = []
placebo_mixed_messages = []
if "trial_scr_neural" not in globals() or not isinstance(trial_scr_neural, pd.DataFrame) or trial_scr_neural.empty:
    display(Markdown("_Placebo-only mixed model could not be fit because `trial_scr_neural` is unavailable._"))
elif "_fit_trialwise_mixedlm" not in globals() or not callable(_fit_trialwise_mixedlm):
    display(Markdown("_Placebo-only mixed model could not be fit because the mixed-model helper was not defined. Run section 6b first._"))
elif "Drug" not in trial_scr_neural.columns:
    display(Markdown("_Placebo-only mixed model could not be fit because the trial-wise table has no `Drug` column._"))
else:
    placebo_df = trial_scr_neural[trial_scr_neural["Drug"].astype(str).str.lower().eq("placebo")].copy()
    display(Markdown(f"Placebo trial rows: **{len(placebo_df)}**; subjects: **{placebo_df['sub_ID'].nunique() if 'sub_ID' in placebo_df.columns else 'NA'}**"))
    if placebo_df.empty:
        display(Markdown("_No placebo-only rows were available for the placebo-focused mixed model._"))
    else:
        placebo_formula = "SCR_Z ~ Neural_Z * C(Group) * C(Domain) + Trial_Z"
        table, msg = _fit_trialwise_mixedlm(placebo_df, "Placebo only: combined safety/threat", placebo_formula)
        if not table.empty:
            placebo_mixed_tables.append(table)
        if msg:
            placebo_mixed_messages.append(msg)
        for domain, domain_df in placebo_df.groupby("Domain"):
            domain_formula = "SCR_Z ~ Neural_Z * C(Group) + Trial_Z"
            table, msg = _fit_trialwise_mixedlm(domain_df, f"Placebo only: {domain} only", domain_formula)
            if not table.empty:
                placebo_mixed_tables.append(table)
            if msg:
                placebo_mixed_messages.append(msg)
        if placebo_mixed_tables:
            placebo_mixed_effects_table = pd.concat(placebo_mixed_tables, ignore_index=True)
            display_df(placebo_mixed_effects_table, rows=120)
            placebo_sig_mixed_effects = placebo_mixed_effects_table[pd.to_numeric(placebo_mixed_effects_table["p"], errors="coerce") < 0.05].copy()
            display(Markdown("**Significant placebo-only mixed-effects terms (p < .05)**"))
            if placebo_sig_mixed_effects.empty:
                display(Markdown("_No placebo-only mixed-effects terms reached p < .05._"))
            else:
                display_df(placebo_sig_mixed_effects, rows=80)
        else:
            placebo_mixed_effects_table = pd.DataFrame()
            display(Markdown("_Placebo-only mixed-effects model could not be fit._"))
        for msg in placebo_mixed_messages:
            display(Markdown(f"_Placebo mixed model note: {msg}_"))


## 7. Decision Boundary Characteristics
Decision-boundary and self-network uncertainty statistics are listed and plotted from saved results where available.


In [ ]:
r15 = results.get("stage15_decision", {})
res14 = r15.get("results_14_self", r15) if isinstance(r15, dict) else {}
df_sad_stats = res14.get('df_sad') if isinstance(res14, dict) else None
df_hc_stats = res14.get('df_hc') if isinstance(res14, dict) else None

def _bh_fdr(pvals):
    p = np.asarray(pvals, dtype=float)
    q = np.full(p.shape, np.nan, dtype=float)
    ok = np.isfinite(p)
    if not ok.any():
        return q
    idx = np.where(ok)[0]
    order = idx[np.argsort(p[ok])]
    ranked = p[order]
    m = len(ranked)
    adj = ranked * m / np.arange(1, m + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    q[order] = np.minimum(adj, 1.0)
    return q

def _cohens_d_independent(a, b):
    a = pd.to_numeric(pd.Series(a), errors='coerce').dropna().to_numpy(dtype=float)
    b = pd.to_numeric(pd.Series(b), errors='coerce').dropna().to_numpy(dtype=float)
    if len(a) < 2 or len(b) < 2:
        return np.nan
    pooled = np.sqrt(((len(a) - 1) * np.var(a, ddof=1) + (len(b) - 1) * np.var(b, ddof=1)) / (len(a) + len(b) - 2))
    return (np.mean(a) - np.mean(b)) / pooled if pooled > 0 else np.nan

def _p_text(p):
    if pd.isna(p):
        return 'NA'
    p = float(p)
    return '<.001' if p < 0.001 else f'={p:.3f}'.replace('0.', '.')

if isinstance(df_sad_stats, pd.DataFrame) and isinstance(df_hc_stats, pd.DataFrame):
    df_plot = pd.concat([df_sad_stats.assign(Group='SAD'), df_hc_stats.assign(Group='HC')], ignore_index=True)
    metric_specs = [
        ('entropy', measure_label('entropy')),
        ('kurtosis', measure_label('kurtosis')),
        ('variance', measure_label('variance')),
        ('boundary_separation', measure_label('boundary_separation')),
        ('decision_margin_css', measure_label('decision_margin_css')),
        ('decision_margin_all', measure_label('decision_margin_all')),
        ('p_csr_css', measure_label('p_csr_css')),
        ('p_csr_csr', measure_label('p_csr_csr')),
    ]
    rows = []
    for metric, label in metric_specs:
        if metric not in df_sad_stats.columns or metric not in df_hc_stats.columns:
            continue
        sad = pd.to_numeric(df_sad_stats[metric], errors='coerce').dropna()
        hc = pd.to_numeric(df_hc_stats[metric], errors='coerce').dropna()
        if len(sad) < 2 or len(hc) < 2:
            continue
        t_val, p_val = stats.ttest_ind(sad, hc, equal_var=False, nan_policy='omit')
        rows.append({
            'metric': metric,
            'label': label,
            'SAD_mean': float(sad.mean()),
            'HC_mean': float(hc.mean()),
            'SAD_sd': float(sad.std(ddof=1)),
            'HC_sd': float(hc.std(ddof=1)),
            't': float(t_val),
            'p': float(p_val),
            'cohens_d_SAD_minus_HC': float(_cohens_d_independent(sad, hc)),
        })
    stats_table = pd.DataFrame(rows)
    if not stats_table.empty:
        stats_table['p_fdr'] = _bh_fdr(stats_table['p'].to_numpy())
    all_p_sad = np.concatenate([np.asarray(p, dtype=float) for p in df_sad_stats['probabilities'].values if len(p) > 0]) if 'probabilities' in df_sad_stats.columns else np.array([])
    all_p_hc = np.concatenate([np.asarray(p, dtype=float) for p in df_hc_stats['probabilities'].values if len(p) > 0]) if 'probabilities' in df_hc_stats.columns else np.array([])
    if all_p_sad.size and all_p_hc.size:
        ks_stat, p_ks = stats.ks_2samp(all_p_sad, all_p_hc)
    else:
        ks_stat, p_ks = np.nan, np.nan
    print("\n" + "="*62)
    print("  DECISION-BOUNDARY SUMMARY: SAD vs HC")
    print("  Welch tests; p_fdr is BH-FDR across scalar boundary metrics")
    print("="*62)
    if not stats_table.empty:
        for _, row in stats_table.iterrows():
            print(f"  {row['label']:<55} t = {row['t']:.4f}, p = {row['p']:.4f}, q = {row['p_fdr']:.4f}, d = {row['cohens_d_SAD_minus_HC']:.3f}")
    print(f"  probability distribution KS     D = {ks_stat:.4f}, p = {p_ks:.4f}")
    print("="*62 + "\n")
    display_df(add_measure_labels(stats_table), rows=30)
    display_df(pd.DataFrame([{'metric': 'probability_distribution_KS', 'D': ks_stat, 'p': p_ks}]), rows=5)
    sns.set_context("talk")
    fig, axes = plt.subplots(3, 3, figsize=(24, 18))
    axes_flat = axes.ravel()
    for ax, (metric, label) in zip(axes_flat, metric_specs):
        if metric not in df_plot.columns:
            ax.axis('off')
            continue
        sns.boxplot(data=df_plot, x='Group', y=metric, hue='Group', palette={'SAD': '#c44e52', 'HC': '#4c72b0'}, legend=False, ax=ax, width=0.55, fliersize=0)
        sns.stripplot(data=df_plot, x='Group', y=metric, hue='Group', palette={'SAD': '#8f2f34', 'HC': '#2e557f'}, legend=False, ax=ax, dodge=False, alpha=0.65, size=5)
        row = stats_table.loc[stats_table['metric'] == metric]
        if not row.empty:
            p_val = float(row.iloc[0]['p'])
            q_val = float(row.iloc[0]['p_fdr'])
            sig = '*' if q_val < 0.05 else ''
            ax.set_title(f"{label}\np{_p_text(p_val)}, q{_p_text(q_val)}{sig}")
        else:
            ax.set_title(label)
        ax.set_xlabel('')
    ax = axes_flat[len(metric_specs)]
    if all_p_sad.size and all_p_hc.size:
        sns.kdeplot(all_p_sad, fill=True, label='SAD', ax=ax, bw_adjust=1.0, color='#c44e52')
        sns.kdeplot(all_p_hc, fill=True, label='HC', ax=ax, bw_adjust=1.0, color='#4c72b0')
        ax.set_title(f"Neural Decision Density\nKS p{_p_text(p_ks)}")
        ax.set_xlabel("P(Threat / CSR)")
        ax.set_xlim(0, 1)
        ax.legend()
    else:
        ax.axis('off')
    fig.suptitle("Analysis 1.4 Decision Boundary Characteristics", y=1.01, fontsize=24)
    fig.tight_layout()
    save_current_fig("memoryfearnetwork_stage13_decision_extended_measures_pLess0.01.png")
    plt.show()
else:
    display(Markdown("_Decision-stat dataframes not found._"))


## 8. Safety Restoration and Threat Discrimination
This section summarizes later-stage safety restoration and threat discrimination outputs.


In [ ]:
r16 = results.get("stage16_safety_threat", {})
res21 = r16.get("results_21_pv") or r16.get("results_21") or r16
df_topo_pv = res21.get("df") if isinstance(res21, dict) else None
p_safe = res21.get("p_safe", np.nan) if isinstance(res21, dict) else np.nan
p_threat = res21.get("p_threat", np.nan) if isinstance(res21, dict) else np.nan
display(Markdown("**Reference Figure: Analysis 2.1 Safety Restoration & Threat Discrimination**"))
display_df(flatten_result_dict(res21), rows=40)
if isinstance(df_topo_pv, pd.DataFrame) and not df_topo_pv.empty:
    display_df(df_topo_pv, rows=20)
    print(f"Safety restoration Group x Drug interaction p = {p_safe:.6g}")
    print(f"Threat discrimination Group x Drug interaction p = {p_threat:.6g}")
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    pal_group = {'SAD': '#c44e52', 'HC': '#4c72b0'}
    def plot_pv_metric(ax, y_col, title, ylabel, p_val):
        sns.pointplot(data=df_topo_pv, x='Drug', y=y_col, hue='Group', palette=pal_group, order=['Placebo', 'Oxytocin'], hue_order=['SAD', 'HC'], dodge=0.2, markers=['o', 's'], capsize=0.1, errorbar='se', ax=ax)
        sns.stripplot(data=df_topo_pv, x='Drug', y=y_col, hue='Group', palette=pal_group, order=['Placebo', 'Oxytocin'], hue_order=['SAD', 'HC'], dodge=True, alpha=0.3, jitter=True, legend=False, ax=ax)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        label = f"Interaction: p={p_val:.3f}" + ("*" if pd.notna(p_val) and p_val < 0.05 else "")
        ax.text(0.5, 0.95, label, transform=ax.transAxes, ha='center', color=('red' if pd.notna(p_val) and p_val < 0.05 else 'black'), fontweight=('bold' if pd.notna(p_val) and p_val < 0.05 else 'normal'))
    safety_col = 'Dist_Safety_PV' if 'Dist_Safety_PV' in df_topo_pv.columns else next((c for c in df_topo_pv.columns if 'Safety' in c), None)
    threat_col = 'Dist_Threat_PV' if 'Dist_Threat_PV' in df_topo_pv.columns else next((c for c in df_topo_pv.columns if 'Threat' in c), None)
    if safety_col:
        plot_pv_metric(axes[0], safety_col, "A. Safety Restoration (PV)\n(CSS vs CS-)", "PV Correlation Dist (Lower = Better)", p_safe)
    if threat_col:
        plot_pv_metric(axes[1], threat_col, "B. Threat Discrimination (PV)\n(CSR vs CSS)", "PV Correlation Dist (Higher = Better)", p_threat)
    save_current_fig("memoryfearnetwork_analysis21_safety_threat_pv_pLess0.01.png")
    plt.show()
else:
    display(Markdown("_No Analysis 2.1 dataframe found._"))


## 9. Drift Efficiency for Safety and Threat Maintenance
This section reports drift-efficiency statistics and plots saved or reconstructed from the stage outputs.


In [ ]:
r18 = results.get("stage18_drift_efficiency", {})
res22 = r18.get("results_22") if isinstance(r18, dict) and isinstance(r18.get("results_22"), dict) else r18
df_drift = res22.get("df") if isinstance(res22, dict) else None
stats_results = res22.get("stats", {}) if isinstance(res22, dict) else {}
display(Markdown("**Reference Figure: Analysis 2.2 Drift Efficiency**"))
display_df(flatten_result_dict(res22), rows=40)
if isinstance(df_drift, pd.DataFrame) and not df_drift.empty:
    display_df(df_drift, rows=20)
    stats_table = pd.DataFrame([{"Domain_Metric": k, "interaction_p": v} for k, v in stats_results.items()])
    display_df(stats_table, rows=20)
    print("Interaction p-values:")
    for k, v in stats_results.items():
        print(f"  {k}: p={v:.6g}")
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    pal_group = {'SAD': '#c44e52', 'HC': '#4c72b0'}
    def _metric_col(metric):
        return metric if metric in df_drift.columns else metric.lower() if metric.lower() in df_drift.columns else None
    def plot_interaction(ax, domain, metric, p_val):
        mcol = _metric_col(metric)
        if mcol is None:
            ax.axis('off'); ax.set_title(f"Missing {metric}"); return
        sub = df_drift[df_drift['Domain'].astype(str).str.lower() == domain.lower()]
        sns.pointplot(data=sub, x='Drug', y=mcol, hue='Group', palette=pal_group, order=['Placebo', 'Oxytocin'], hue_order=['SAD', 'HC'], dodge=0.2, markers=['o', 's'], capsize=0.1, errorbar='se', ax=ax)
        ax.set_title(f"{domain}: {metric}\nInteraction p={p_val:.3f}")
    plot_interaction(axes[0,0], "Safety", "Cosine", stats_results.get("Safety_Cosine", np.nan))
    plot_interaction(axes[0,1], "Safety", "Projection", stats_results.get("Safety_Projection", np.nan))
    plot_interaction(axes[1,0], "Threat", "Cosine", stats_results.get("Threat_Cosine", np.nan))
    plot_interaction(axes[1,1], "Threat", "Projection", stats_results.get("Threat_Projection", np.nan))
    save_current_fig("memoryfearnetwork_analysis22_drift_efficiency_pLess0.01.png")
    plt.show()
else:
    display(Markdown("_No drift-efficiency dataframe found._"))


## 9b. Second Threat Metric: Shock-Target Drift Efficiency

In [ ]:
# Second threat metric visualization: drift efficiency toward shock/US target pattern.
def _find_drift_payload(obj, max_depth=8):
    if max_depth < 0:
        return None
    if isinstance(obj, dict):
        df = obj.get("df")
        if isinstance(df, pd.DataFrame) and "Domain" in df.columns and df["Domain"].astype(str).str.contains("Shock", case=False, na=False).any():
            return obj
        for val in obj.values():
            found = _find_drift_payload(val, max_depth - 1)
            if found is not None:
                return found
    elif isinstance(obj, (list, tuple)):
        for val in obj:
            found = _find_drift_payload(val, max_depth - 1)
            if found is not None:
                return found
    return None

drift_payload = _find_drift_payload(results) if isinstance(results, dict) else None
if drift_payload is None:
    display(Markdown("_No shock-target drift-efficiency rows found yet. Re-run drift efficiency after the shock-target update._"))
else:
    df_drift_shock_source = drift_payload.get("df")
    drift_stats = drift_payload.get("stats", {}) if isinstance(drift_payload, dict) else {}
    shock_labels = drift_payload.get("shock_target_labels", []) if isinstance(drift_payload, dict) else []
    shock_rows = df_drift_shock_source[df_drift_shock_source["Domain"].astype(str).str.contains("Shock", case=False, na=False)].copy()
    display(Markdown(f"**Shock-target labels used by the `.py` script:** `{shock_labels}`"))
    display(Markdown("**Threat shock-target drift-efficiency rows**"))
    display_df(shock_rows, rows=50)
    shock_stat_rows = []
    for key, val in drift_stats.items():
        if "Shock" in str(key):
            shock_stat_rows.append({"Domain_Metric": key, "interaction_p": val})
    shock_stats_table = pd.DataFrame(shock_stat_rows)
    display(Markdown("**Threat shock-target drift-efficiency statistics**"))
    display_df(shock_stats_table, rows=20)
    if not shock_rows.empty:
        sns.set_context("poster", font_scale=0.85)
        fig, axes = plt.subplots(1, 2, figsize=(16, 6), squeeze=False)
        pal_group = {"SAD": "#c44e52", "HC": "#4c72b0"}
        for ax, metric in zip(axes.ravel(), ["Cosine", "Projection"]):
            mcol = metric if metric in shock_rows.columns else metric.lower() if metric.lower() in shock_rows.columns else None
            if mcol is None:
                ax.axis("off")
                ax.set_title(f"Missing {metric}")
                continue
            p_val = drift_stats.get(f"Threat Shock Target_{metric}", np.nan)
            sns.pointplot(data=shock_rows, x="Drug", y=mcol, hue="Group", palette=pal_group, order=["Placebo", "Oxytocin"], hue_order=["SAD", "HC"], dodge=0.2, markers=["o", "s"], capsize=0.1, errorbar="se", ax=ax)
            sns.stripplot(data=shock_rows, x="Drug", y=mcol, hue="Group", palette=pal_group, order=["Placebo", "Oxytocin"], hue_order=["SAD", "HC"], dodge=True, alpha=0.35, jitter=True, legend=False, ax=ax)
            ax.axhline(0, color="gray", linestyle="--", linewidth=1)
            ax.set_title(f"Threat Shock Target: {metric}\nGroup x Drug p={p_val:.3g}" if np.isfinite(p_val) else f"Threat Shock Target: {metric}")
            handles, labels = ax.get_legend_handles_labels()
            dedup = {}
            for handle, label in zip(handles, labels):
                if label not in dedup:
                    dedup[label] = handle
            ax.legend(list(dedup.values()), list(dedup.keys()), title="Group", loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True, borderaxespad=0)
        fig.tight_layout(w_pad=4.0)
        save_current_fig(f"{PIPELINE_NAME.lower()}_threat_shock_target_drift_efficiency.png")
        plt.show()


## 10. Probabilistic Opening / Decision-Probability Extraction
This section summarizes probabilistic opening and decision-probability extraction outputs.


In [ ]:
r19 = results.get("stage19_prob_opening", {})
res23 = r19.get("results_23") if isinstance(r19, dict) and isinstance(r19.get("results_23"), dict) else r19
df_metrics = res23.get("df") if isinstance(res23, dict) else None
stats_results = dict(res23.get("stats", {})) if isinstance(res23, dict) and isinstance(res23.get("stats", {}), dict) else {}

# Match Section 7 decision-boundary measures when they are available in saved stage 15 outputs.
r15_for_opening = results.get("stage15_decision", {})
res14_for_opening = r15_for_opening.get("results_14_self", r15_for_opening) if isinstance(r15_for_opening, dict) else {}
df_sad_decision = res14_for_opening.get('df_sad') if isinstance(res14_for_opening, dict) else None
df_hc_decision = res14_for_opening.get('df_hc') if isinstance(res14_for_opening, dict) else None
section7_to_section10 = {
    'entropy': 'Entropy',
    'kurtosis': 'Kurtosis',
    'variance': 'Variance',
    'boundary_separation': 'Boundary Separation',
    'decision_margin_css': 'CSS Decision Margin',
    'decision_margin_all': 'Overall Decision Margin',
    'p_csr_css': 'P(CSR) CSS',
    'p_csr_csr': 'P(CSR) CSR',
}
measure_order = list(section7_to_section10.values())
measure_display_order = [measure_label(met) for met in measure_order]

def _interaction_p_lm(df, metric):
    sub = df[['Group', 'Drug', metric]].copy()
    sub[metric] = pd.to_numeric(sub[metric], errors='coerce')
    sub = sub.dropna()
    if sub['Group'].nunique() != 2 or sub['Drug'].nunique() != 2 or len(sub) < 8:
        return np.nan
    group_levels = sorted(sub['Group'].dropna().unique())
    drug_levels = sorted(sub['Drug'].dropna().unique())
    g = (sub['Group'] == group_levels[-1]).astype(float).to_numpy()
    d = (sub['Drug'] == drug_levels[-1]).astype(float).to_numpy()
    y = sub[metric].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(sub)), g, d, g * d])
    try:
        beta, residuals, rank, svals = np.linalg.lstsq(X, y, rcond=None)
        df_resid = len(y) - rank
        if df_resid <= 0:
            return np.nan
        rss = float(np.sum((y - X @ beta) ** 2))
        sigma2 = rss / df_resid
        cov = sigma2 * np.linalg.pinv(X.T @ X)
        se = float(np.sqrt(cov[3, 3]))
        if not np.isfinite(se) or se == 0:
            return np.nan
        t_val = float(beta[3] / se)
        return float(2 * stats.t.sf(abs(t_val), df_resid))
    except Exception:
        return np.nan

if isinstance(df_metrics, pd.DataFrame) and not df_metrics.empty:
    df_metrics = df_metrics.copy()
    native_to_display = {
        'P_CSR_CSS': 'P(CSR) CSS',
        'P_CSR_CSR': 'P(CSR) CSR',
        'Boundary_Separation': 'Boundary Separation',
        'Decision_Margin_CSS': 'CSS Decision Margin',
        'Decision_Margin_All': 'Overall Decision Margin',
    }
    for native_col, display_col in native_to_display.items():
        if native_col in df_metrics.columns and display_col not in df_metrics.columns:
            df_metrics[display_col] = df_metrics[native_col]
    if isinstance(df_sad_decision, pd.DataFrame) and isinstance(df_hc_decision, pd.DataFrame):
        decision_df = pd.concat([df_sad_decision.assign(Group='SAD'), df_hc_decision.assign(Group='HC')], ignore_index=True)
        rename_cols = {'sub': 'Subject'}
        rename_cols.update(section7_to_section10)
        decision_df = decision_df.rename(columns=rename_cols)
        merge_cols = ['Subject'] + [col for col in measure_order if col in decision_df.columns]
        # Keep Section 10 native Entropy/Kurtosis/Variance if present; add only missing/expanded measures from Section 7.
        add_cols = ['Subject'] + [col for col in merge_cols if col != 'Subject' and col not in df_metrics.columns]
        if len(add_cols) > 1:
            df_metrics = df_metrics.merge(decision_df[add_cols].drop_duplicates('Subject'), on='Subject', how='left')
    display(Markdown("**Reference Figure: Analysis 2.3 Probabilistic Opening / Decision-Probability Extraction**"))
    display(Markdown("Expanded to the same scalar decision-boundary measure set used in Section 7 when saved measures are available."))
    display_df(flatten_result_dict(res23), rows=40)
    display_df(df_metrics.rename(columns={c: measure_label(c) for c in df_metrics.columns if c not in {'Subject', 'Group', 'Drug'}}), rows=25)
    stat_rows = []
    for met in measure_order:
        if met not in df_metrics.columns:
            continue
        p_interaction = stats_results.get(met, _interaction_p_lm(df_metrics, met))
        stat_rows.append({'measure': met, 'measure_label': measure_label(met), 'group_x_drug_interaction_p': p_interaction, 'n_nonmissing': int(pd.to_numeric(df_metrics[met], errors='coerce').notna().sum())})
        print(f"{measure_label(met)} Group x Drug interaction p = {p_interaction:.6g}" if np.isfinite(p_interaction) else f"{measure_label(met)} Group x Drug interaction p = NA")
    stat_table = pd.DataFrame(stat_rows)
    display_df(stat_table, rows=30)
    plot_measures = [met for met in measure_order if met in df_metrics.columns]
    n_cols = 3
    n_rows = int(np.ceil(len(plot_measures) / n_cols)) if plot_measures else 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(24, 6 * n_rows), squeeze=False)
    for ax, met in zip(axes.ravel(), plot_measures):
        pval_series = stat_table.loc[stat_table['measure'] == met, 'group_x_drug_interaction_p']
        pval = float(pval_series.iloc[0]) if len(pval_series) and pd.notna(pval_series.iloc[0]) else np.nan
        sns.boxplot(data=df_metrics, x='Group', y=met, hue='Drug', ax=ax)
        sns.stripplot(data=df_metrics, x='Group', y=met, hue='Drug', dodge=True, ax=ax, alpha=0.45, size=4, legend=False, color='black')
        ax.set_title(f"{measure_label(met)}\nGroup x Drug p={pval:.3f}" if np.isfinite(pval) else f"{measure_label(met)}\nGroup x Drug p=NA")
        ax.set_xlabel('')
        if ax.get_legend() is not None:
            ax.get_legend().set_title('Drug')
    for ax in axes.ravel()[len(plot_measures):]:
        ax.axis('off')
    fig.suptitle("Analysis 2.3 Probabilistic Opening: Section 7 Measure Set", y=1.01, fontsize=24)
    fig.tight_layout()
    save_current_fig("memoryfearnetwork_analysis23_probabilistic_opening_extended_measures_pLess0.01.png")
    plt.show()
else:
    display(Markdown("_No probabilistic-opening dataframe found._"))


## 11. Spatial Re-Alignment
Spatial re-alignment statistics and figures are displayed here.


In [ ]:
r20 = results.get("stage20_realignment", {})
res24 = r20.get("results_24") if isinstance(r20, dict) and isinstance(r20.get("results_24"), dict) else r20
acc_plc = np.asarray(res24.get("acc_plc", []), dtype=float) if isinstance(res24, dict) else np.array([])
acc_oxt = np.asarray(res24.get("acc_oxt", []), dtype=float) if isinstance(res24, dict) else np.array([])
p_val = float(res24.get("p_val", np.nan)) if isinstance(res24, dict) else np.nan
display(Markdown("**Reference Figure: Analysis 2.4 Spatial Re-Alignment**"))
display_df(flatten_result_dict(res24), rows=40)
if acc_plc.size and acc_oxt.size:
    m_plc, m_oxt = np.nanmean(acc_plc), np.nanmean(acc_oxt)
    print(f"Accuracy (Train HC -> Test SAD-Placebo): {m_plc:.3f}; n={acc_plc.size}")
    print(f"Accuracy (Train HC -> Test SAD-Oxytocin): {m_oxt:.3f}; n={acc_oxt.size}")
    print(f"OXT > Placebo p = {p_val:.6g}")
    sig_label = "*" if p_val < 0.05 else "ns"
    fig, ax = plt.subplots(figsize=(10, 5))
    matrix_data = np.array([[m_plc, m_oxt]])
    annot_data = np.array([[f"{m_plc:.3f}", f"{m_oxt:.3f}\n({sig_label})"]])
    sns.heatmap(matrix_data, annot=annot_data, fmt="", cmap="RdBu_r", vmin=0.3, vmax=0.7, center=0.5, cbar=True, xticklabels=['Test: SAD-Placebo', 'Test: SAD-Oxytocin'], yticklabels=['Train: HC-Placebo (Anal 1.1)'], ax=ax)
    ax.set_title(f"Analysis 2.4: Spatial Re-Alignment\n(OXT vs PLC Improvement: p={p_val:.3f})")
    plt.yticks(rotation=0)
    save_current_fig("memoryfearnetwork_analysis24_spatial_realignment_pLess0.01.png")
    plt.show()
else:
    display(Markdown("_No spatial realignment accuracy arrays found._"))


## 12. Reverse Cross-Decoding
Reverse cross-decoding statistics and figures are displayed here.


In [ ]:
r21 = results.get("stage21_reverse", {})
res25 = r21.get("results_25") if isinstance(r21, dict) and isinstance(r21.get("results_25"), dict) else r21
acc_plc = np.asarray(res25.get("acc_plc", []), dtype=float) if isinstance(res25, dict) else np.array([])
acc_oxt = np.asarray(res25.get("acc_oxt", []), dtype=float) if isinstance(res25, dict) else np.array([])
p_val = float(res25.get("p_val", np.nan)) if isinstance(res25, dict) else np.nan
display(Markdown("**Reference Figure: Analysis 2.5 Reverse Cross-Decoding**"))
display_df(flatten_result_dict(res25), rows=40)
if acc_plc.size and acc_oxt.size:
    m_plc, m_oxt = np.nanmean(acc_plc), np.nanmean(acc_oxt)
    print(f"Accuracy (Train SAD -> Test HC-Placebo): {m_plc:.3f}; n={acc_plc.size}")
    print(f"Accuracy (Train SAD -> Test HC-Oxytocin): {m_oxt:.3f}; n={acc_oxt.size}")
    print(f"OXT > Placebo p = {p_val:.6g}")
    sig_label = "*" if p_val < 0.05 else "ns"
    fig, ax = plt.subplots(figsize=(10, 5))
    matrix_data = np.array([[m_plc, m_oxt]])
    annot_data = np.array([[f"{m_plc:.3f}", f"{m_oxt:.3f}\n({sig_label})"]])
    sns.heatmap(matrix_data, annot=annot_data, fmt="", cmap="RdBu_r", vmin=0.3, vmax=0.7, center=0.5, cbar=True, xticklabels=['Test: HC-Placebo', 'Test: HC-Oxytocin'], yticklabels=['Train: SAD-Placebo (Anal 1.1)'], ax=ax)
    ax.set_title(f"Analysis 2.5: Reverse Cross-Decoding\n(OXT vs PLC Improvement: p={p_val:.3f})")
    plt.yticks(rotation=0)
    save_current_fig("memoryfearnetwork_analysis25_reverse_cross_decoding_pLess0.01.png")
    plt.show()
else:
    display(Markdown("_No reverse cross-decoding accuracy arrays found._"))


## 13. Group-Wise Neural-Clinical Pearson Correlations
Pearson correlation tables and heatmaps are shown for group-wise neural-clinical associations.


In [ ]:
r27 = results.get("stage27_pearson", {})
r26 = results.get("stage26_master_merge", {})
df_master = r26.get("df_master_analysis") if isinstance(r26, dict) else None
df_res_grp = r27.get("df_res_grp") if isinstance(r27, dict) else None
neural_metrics = list(r27.get("neural_metrics", [])) if isinstance(r27, dict) else []
clinical_indices = list(r27.get("clinical_indices", [])) if isinstance(r27, dict) else []
if not neural_metrics and isinstance(df_res_grp, pd.DataFrame):
    neural_metrics = list(df_res_grp["Neural"].dropna().unique())
if not clinical_indices and isinstance(df_res_grp, pd.DataFrame):
    clinical_indices = list(df_res_grp["Clinical"].dropna().unique())
display(Markdown("**Reference Figures: Group-wise neural-clinical Pearson regression grids**"))
display_df(add_measure_labels(df_res_grp), rows=80)
if isinstance(df_master, pd.DataFrame) and isinstance(df_res_grp, pd.DataFrame) and not df_res_grp.empty:
    group_col = "Group" if "Group" in df_master.columns else "Group_x" if "Group_x" in df_master.columns else None
    print(f"{'Group':<6} | {'Neural measure':<55} | {'Clinical outcome':<22} | {'r':<7} | {'p':<10} | Sig")
    print("-" * 100)
    for _, row in df_res_grp.iterrows():
        print(f"{row['Group']:<6} | {measure_label(row['Neural']):<55} | {measure_label(row['Clinical']):<22} | {row['r']:<7.3f} | {row['p']:<10.5f} | {row['sig']}")
    plot_neural = neural_metrics[:]
    plot_clinical = clinical_indices[:]
    for n_m in plot_neural:
        if n_m not in df_master.columns:
            continue
        fig, axes = plt.subplots(1, len(plot_clinical), figsize=(max(8, 5.2 * len(plot_clinical)), 6))
        axes = np.atleast_1d(axes)
        for j, c_i in enumerate(plot_clinical):
            ax = axes[j]
            if c_i not in df_master.columns or group_col is None:
                ax.axis('off'); continue
            for grp in sorted(df_master[group_col].dropna().unique()):
                grp_data = df_master[df_master[group_col] == grp]
                stats_row = df_res_grp[(df_res_grp['Group'] == grp) & (df_res_grp['Neural'] == n_m) & (df_res_grp['Clinical'] == c_i)]
                label_str = str(grp)
                if not stats_row.empty:
                    sig = stats_row['sig'].values[0]
                    r = stats_row['r'].values[0]
                    label_str = f"{grp} (r={r:.2f}{'' if sig=='ns' else sig})"
                sns.regplot(data=grp_data, x=n_m, y=c_i, ax=ax, label=label_str, scatter_kws={'alpha': 0.45})
            ax.set_xlabel(measure_label(n_m))
            ax.set_ylabel(measure_label(c_i))
            ax.set_title(measure_label(c_i))
            ax.legend(fontsize=8, loc='best')
            sns.despine(ax=ax)
        plt.suptitle(f"Group-wise associations: {measure_label(n_m)}", fontsize=16, y=1.02)
        save_current_fig(f"memoryfearnetwork_pearson_grid_pLess0.01_{n_m}.png")
        plt.show()
else:
    display(Markdown("_Pearson results or master dataframe not found._"))


## 16. Outlier Removal and Z-Scoring
This section audits outlier removal and z-scoring for neural, clinical, and covariate variables.


In [ ]:
r26 = results.get("stage26_master_merge", {})
r29 = results.get("stage29_zscore_outlier", {})
df_master = r26.get("df_master_analysis") if isinstance(r26, dict) else None
base_pearson = results.get("stage27_pearson", {})
neural_metrics = list(base_pearson.get("neural_metrics", [])) if isinstance(base_pearson, dict) else []
clinical_indices = list(base_pearson.get("clinical_indices", [])) if isinstance(base_pearson, dict) else []
covariates = ['demo_age']
z_limit = float(r29.get('z_limit', 3.0)) if isinstance(r29, dict) else 3.0
display(Markdown("**Reference Audit: Outlier removal and final z-scoring**"))
if isinstance(df_master, pd.DataFrame):
    if not neural_metrics:
        neural_metrics = [c for c in df_master.columns if c.startswith('Neural_') and not c.endswith('_z')]
    if not clinical_indices:
        clinical_indices = [c for c in ['lsas_total','lsas_fear','lsas_avoid','dass_anxiety','dass_stress','ecr_total'] if c in df_master.columns]
    df_z = df_master.copy()
    rows = []
    for col in neural_metrics + clinical_indices + covariates:
        if col in df_z.columns:
            vals = pd.to_numeric(df_z[col], errors='coerce')
            init_z = (vals - vals.mean(skipna=True)) / vals.std(skipna=True, ddof=0)
            outlier_mask = init_z.abs() > z_limit
            n_removed = int(outlier_mask.sum())
            df_z.loc[outlier_mask, col] = np.nan
            vals2 = pd.to_numeric(df_z[col], errors='coerce')
            df_z[f'{col}_z'] = (vals2 - vals2.mean(skipna=True)) / vals2.std(skipna=True, ddof=0)
            rows.append({'variable': col, 'variable_label': measure_label(col), 'n_removed': n_removed, 'n_valid_after': int(vals2.notna().sum()), 'mean_after': float(vals2.mean(skipna=True)), 'sd_after': float(vals2.std(skipna=True, ddof=1))})
    summary = pd.DataFrame(rows)
    display_df(summary, rows=80)
    print(f"Outlier threshold: absolute z > {z_limit}")
    for _, row in summary.iterrows():
        if row['n_removed'] > 0:
            print(f"{row['variable_label']}: removed {int(row['n_removed'])} outliers")
    plot_cols = [f'{c}_z' for c in neural_metrics[:8] + clinical_indices[:8] if f'{c}_z' in df_z.columns]
    if plot_cols:
        fig, ax = plt.subplots(figsize=(max(10, 0.55 * len(plot_cols)), 5))
        sns.boxplot(data=df_z[plot_cols].rename(columns={c: measure_label(c) for c in plot_cols}), ax=ax, color='#9C755F')
        ax.set_title('Z-scored neural, clinical, and covariate variables after outlier removal')
        ax.tick_params(axis='x', rotation=60)
        save_current_fig('memoryfearnetwork_outlier_zscore_boxplots_pLess0.01.png')
        plt.show()
else:
    display(Markdown("_Master dataframe not found for z-scoring audit._"))


## 17. Z-Scored OLS / Regression Plots for Neural-Clinical Associations
This final section lists z-scored OLS models and plots regression coefficients or saved regression figures where available.


In [ ]:
r26 = results.get("stage26_master_merge", {})
base_pearson = results.get("stage27_pearson", {})
df_master = r26.get("df_master_analysis") if isinstance(r26, dict) else None
neural_metrics = list(base_pearson.get("neural_metrics", [])) if isinstance(base_pearson, dict) else []
clinical_indices = list(base_pearson.get("clinical_indices", [])) if isinstance(base_pearson, dict) else []
def _make_z_df(df, variables, z_limit=3.0):
    out = df.copy()
    for col in variables:
        if col in out.columns:
            vals = pd.to_numeric(out[col], errors='coerce')
            z0 = (vals - vals.mean(skipna=True)) / vals.std(skipna=True, ddof=0)
            out.loc[z0.abs() > z_limit, col] = np.nan
            vals2 = pd.to_numeric(out[col], errors='coerce')
            out[f'{col}_z'] = (vals2 - vals2.mean(skipna=True)) / vals2.std(skipna=True, ddof=0)
    return out
display(Markdown("**Reference Figures: Z-scored OLS/regression grids**"))
if isinstance(df_master, pd.DataFrame):
    group_col = "Group" if "Group" in df_master.columns else "Group_x" if "Group_x" in df_master.columns else None
    if not neural_metrics:
        neural_metrics = [c for c in df_master.columns if c.startswith('Neural_') and not c.endswith('_z')]
    if not clinical_indices:
        clinical_indices = [c for c in ['lsas_total','lsas_fear','lsas_avoid','dass_anxiety','dass_stress','ecr_total'] if c in df_master.columns]
    df_z = _make_z_df(df_master, neural_metrics + clinical_indices + ['demo_age'])
    neural_z = [f'{c}_z' for c in neural_metrics if f'{c}_z' in df_z.columns]
    clinical_z = [f'{c}_z' for c in clinical_indices if f'{c}_z' in df_z.columns]
    rows = []
    for n_m in neural_z:
        print(f"\n{'='*80}\nNEURAL PREDICTOR: {measure_label(n_m)}\n{'='*80}")
        fig, axes = plt.subplots(1, len(clinical_z), figsize=(max(8, 5.5 * len(clinical_z)), 6), sharey=True)
        axes = np.atleast_1d(axes)
        for i, c_i in enumerate(clinical_z):
            ax = axes[i]
            print(f"\n--- Clinical Outcome: {measure_label(c_i)} ---")
            for grp in sorted(df_z[group_col].dropna().unique()) if group_col else []:
                analysis_df = df_z[df_z[group_col] == grp][[n_m, c_i]].dropna()
                if len(analysis_df) > 5:
                    X = sm.add_constant(analysis_df[n_m])
                    y = analysis_df[c_i]
                    model = sm.OLS(y, X).fit()
                    beta = float(model.params[n_m])
                    tval = float(model.tvalues[n_m])
                    pval = float(model.pvalues[n_m])
                    sig_note = " *SIGNIFICANT*" if pval < 0.05 else ""
                    print(f"[{grp:<3}] N={len(analysis_df):<3} | Beta={beta:>6.3f} | t={tval:>6.2f} | p={pval:>6.4f}{sig_note}")
                    rows.append({'Group': grp, 'Neural_z': n_m, 'Neural_label': measure_label(n_m), 'Clinical_z': c_i, 'Clinical_label': measure_label(c_i), 'N': len(analysis_df), 'beta': beta, 't': tval, 'p': pval})
                    sns.regplot(data=analysis_df, x=n_m, y=c_i, ax=ax, label=f"{grp} (p={pval:.3f})", scatter_kws={'alpha': 0.4}, line_kws={'lw': 3})
                else:
                    print(f"[{grp:<3}] Insufficient data (N < 5)")
            ax.set_title(measure_label(c_i))
            ax.set_xlabel(measure_label(n_m))
            ax.set_ylabel(measure_label(c_i))
            ax.axhline(0, color='gray', linestyle='--', alpha=0.3)
            ax.axvline(0, color='gray', linestyle='--', alpha=0.3)
            ax.legend(fontsize='small', loc='best')
        plt.suptitle(f"Brain-behavior associations: {measure_label(n_m)} (outliers removed)", fontsize=16, y=1.02)
        sns.despine()
        save_current_fig(f"memoryfearnetwork_z_ols_grid_pLess0.01_{n_m}.png")
        plt.show()
    df_ols = pd.DataFrame(rows)
    display_df(df_ols, rows=80)
else:
    display(Markdown("_Master dataframe not found for z-scored OLS grids._"))
